# Rationale scoring pipeline

Builds the per-trial similarity files the main analysis starts from
(`data/comprehensive_theme_file2.csv`, `comprehensive_language_file2.csv`,
`comprehensive_detail_lengths2.csv`, `comprehensive_stance_file.csv`), consumed by
`HAI_analysis_directionalR_revisionFF_MLM_bootstrap.ipynb`.

- **Theme:** cosine similarity of `all-MiniLM-L6-v2` sentence embeddings
  (first vs final rationale, each vs the AI rationale).
- **Language style:** cosine similarity of LIWC-22 category vectors. LIWC-22 is
  licensed software; its output CSVs are produced outside this notebook.
- **Detail / stance:** word counts and coded stances.

Original scoring scripts by Harsh; combined into this notebook by Sharon. The later
sections are an earlier version of the statistical models, superseded by the main
analysis notebook.

**Inputs are participant data and are not in the repository** (rationale exports
with embeddings and the LIWC-22 result files, expected in the working directory or
`data/`). Outputs are cleared in the committed copy, so keep it that way: run it on a
copy outside the repo, or clear outputs before committing.

## New as of sept 8 after duplicates fixed

## Fixing datasets

In [ ]:
# build_all_conditions_with_embeddings_dual_maps.py
import ast
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

# ---------- 1) ROLE mapping (for scenario/role only) ----------
# (the original mapping you use to label first/final)
COLUMN_MAPPINGS = {
    "baseline": {
        "sexism_explanations": [("Q16","Q18"), ("Q4","Q5"), ("Q20","Q22"), ("Q24","Q26")],
        "aita_explanations":   [("Q8","Q10"),  ("Q28","Q30"),("Q32","Q34"), ("Q36","Q38")],
    },
    "low": {
        "sexism_explanations": [("Q4","Q5"),   ("Q18","Q20"), ("Q24","Q26"), ("Q30","Q32")],
        "aita_explanations":   [("Q8","Q10"),  ("Q38","Q40"), ("Q44.1","Q46"),("Q50","Q52")],
    },
    "high": {
        "sexism_explanations": [("Q4","Q5"),   ("Q2","Q5.1"), ("Q2.1","Q5.2"),("Q2.2","Q5.3")],
        "aita_explanations":   [("Q8","Q10"),  ("Q2.3","Q5.4"),("Q2.4","Q5.5"),("Q2.5","Q5.6")],
    },
}

# ---------- 2) STANCE mapping (pairs: firstQ, finalQ) ----------
STANCE_COLS_BY_SCENARIO = {
    "high": {
        "sexism_explanations": [("Q3","Q6"), ("Q1.1","Q4.1"), ("Q1.2","Q4.2"), ("Q1.3","Q4.3")],
        "aita_explanations":   [("Q7","Q9"), ("Q1.4","Q4.4"), ("Q1.5","Q4.5"), ("Q1.6","Q4.6")],
    },
    "low": {
        "sexism_explanations": [("Q3","Q6"), ("Q17","Q19"), ("Q23","Q25"), ("Q29","Q31")],
        "aita_explanations":   [("Q7","Q9"), ("Q37","Q39"), ("Q43","Q45"), ("Q49.7","Q51")],
    },
    "baseline": {
        "sexism_explanations": [("Q15","Q17"), ("Q3","Q6"), ("Q19","Q21"), ("Q23","Q25")],
        "aita_explanations":   [("Q7","Q9"),  ("Q27","Q29"), ("Q31","Q33"), ("Q35","Q37")],
    },
}

# ---------- utilities ----------
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found. Available: {list(df.columns)}")

def encode_text_series(texts: pd.Series, batch_size=64):
    raw = texts.fillna("").astype(str).tolist()
    mask = [t.strip() != "" for t in raw]
    out = [None] * len(raw)
    if any(mask):
        idx = [i for i, ok in enumerate(mask) if ok]
        vecs = model.encode([raw[i] for i in idx],
                            batch_size=batch_size,
                            convert_to_numpy=True,
                            show_progress_bar=False)
        for j, i in enumerate(idx):
            out[i] = vecs[j].tolist()
    return out

def to_vec(v):
    if v is None:
        return None
    if isinstance(v, np.ndarray):
        a = v.ravel()
        return a if a.size else None
    if isinstance(v, list):
        a = np.asarray(v, dtype=float).ravel()
        return a if a.size else None
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return None
        try:
            a = np.asarray(ast.literal_eval(s), dtype=float).ravel()
            return a if a.size else None
        except Exception:
            return None
    try:
        a = np.asarray(v, dtype=float).ravel()
        return a if a.size else None
    except Exception:
        return None

def cosine_safe(a, b):
    if a is None or b is None or a.size == 0 or b.size == 0:
        return np.nan
    return float(cosine_similarity(a.reshape(1,-1), b.reshape(1,-1))[0,0])

# ---------- mapping builders ----------
def build_role_lookup(condition: str):
    """
    Using COLUMN_MAPPINGS (the original): map colID -> (scenario, role, base, idx)
    """
    cond = condition.lower()
    m = {}
    for category, pairs in COLUMN_MAPPINGS[cond].items():
        base = "sexism" if category.startswith("sexism") else "aita"
        for idx, (q_first, q_final) in enumerate(pairs, start=1):
            scen = f"{base}-{idx}"
            m[str(q_first).strip().lower()] = (scen, "first", base, idx)
            m[str(q_final).strip().lower()] = (scen, "final", base, idx)
    return m

def stance_from_pairs(row, df_row_source: pd.Series, condition: str):
    """
    Using STANCE_COLS_BY_SCENARIO: pick the stance column
      - role == 'first' -> take the first element of the pair
      - role == 'final' -> take the second element of the pair
    """
    base = row.get("base")
    idx  = row.get("idx")
    role = row.get("role")
    if pd.isna(base) or pd.isna(idx) or pd.isna(role):
        return np.nan
    try:
        idx = int(idx)
    except Exception:
        return np.nan

    cond = condition.lower()
    cat_key = "sexism_explanations" if base == "sexism" else "aita_explanations"
    pairs = STANCE_COLS_BY_SCENARIO.get(cond, {}).get(cat_key, [])
    if not (1 <= idx <= len(pairs)):
        return np.nan

    q_first, q_final = pairs[idx-1]
    target_col = q_first if role == "first" else q_final
    # Return stance from original row's columns if present
    return df_row_source.get(target_col, np.nan) if target_col in df_row_source.index else np.nan

# ---------- per-condition processing ----------
def process_condition(path: str, condition: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    colid = pick_col(df, ["colID", "ColumnID"])
    text_col = pick_col(df, ["text", "Text", "content"])
    id_col = pick_col(df, ["ResponseId", "ResponseID", "response_id"])

    # scenario/role from the ORIGINAL role mapping
    role_lookup = build_role_lookup(condition)
    df["colid_norm"] = df[colid].astype(str).str.strip().str.lower()
    mapped = (
        df["colid_norm"]
          .map(lambda q: role_lookup.get(q, (pd.NA, pd.NA, pd.NA, pd.NA)))
          .apply(pd.Series)
          .rename(columns={0:"scenario", 1:"role", 2:"base", 3:"idx"})
    )
    df = pd.concat([df, mapped], axis=1)

    # standardize output columns
    out = df[[id_col, colid, text_col, "scenario", "role", "base", "idx"]].copy()
    out = out.rename(columns={id_col:"response_id", text_col:"text", colid:"ColumnID"})
    out["condition"] = condition

    # stance via STANCE_COLS_BY_SCENARIO
    out["stance"] = out.apply(lambda r: stance_from_pairs(r, df.loc[r.name], condition), axis=1)

    # embeddings for text
    out["embedding"] = encode_text_series(out["text"])

    keep = ["response_id","ColumnID","text","scenario","role","condition","embedding","stance"]
    for c in keep:
        if c not in out.columns:
            out[c] = np.nan
    return out[keep]

# ---------- file paths ----------
P_BASE = "LIWC-22 Results - H-AI_Subjectivity_Study_Baseli___ - LIWC Analysis.csv"
P_LOW  = "LIWC-22 Results - H-AI_Subjectivity_Study_Low_In___ - LIWC Analysis.csv"
P_HIGH = "LIWC-22 Results - H-AI_Subjectivity_Study_High_I___ - LIWC Analysis (1).csv"

# ---------- run ----------
df_base = process_condition(P_BASE, "baseline")
df_low  = process_condition(P_LOW,  "low")
df_high = process_condition(P_HIGH, "high")

df_all = pd.concat([df_base, df_low, df_high], ignore_index=True)

# de-dupe on (condition, response_id, scenario, role)
key = ["condition","response_id","scenario","role"]
df_all = df_all.drop_duplicates(subset=key, keep="first").copy()

# compute cosine_first_final per (condition, response_id, scenario)
df_all["_vec"] = df_all["embedding"].apply(to_vec)

pairs = []
for (cond, rid, scen), g in df_all.groupby(["condition","response_id","scenario"], dropna=False):
    g = g.set_index("role", drop=False)

    # texts (to decide NaN rule)
    t_first = g.loc["first", "text"] if "first" in g.index else None
    t_final = g.loc["final", "text"] if "final" in g.index else None

    # vectors
    v_first = to_vec(g.loc["first", "_vec"]) if "first" in g.index else None
    v_final = to_vec(g.loc["final", "_vec"]) if "final" in g.index else None

    if (t_first is None or str(t_first).strip()=="") and (t_final is None or str(t_final).strip()==""):
        cos_ff = np.nan
    else:
        cos_ff = cosine_safe(v_first, v_final)

    pairs.append({"condition":cond, "response_id":rid, "scenario":scen, "cosine_first_final":cos_ff})

df_cos = pd.DataFrame(pairs)

df_out = (
    df_all.drop(columns=["_vec"])
          .merge(df_cos, on=["condition","response_id","scenario"], how="left", validate="m:1")
)

df_out.to_csv("all_conditions_with_embeddings.csv", index=False)
print("✅ Saved: all_conditions_with_embeddings.csv")
print("Rows:", len(df_out), " | Unique response_ids:", df_out["response_id"].nunique())


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# -------- Load AI explanation files --------
ai_low = pd.read_csv("data/LIWC-22 Results - low_interactivity_AI_explanati___ - LIWC Analysis- wStance.csv")
ai_high = pd.read_csv("data/LIWC-22 Results - AIhigh - LIWC Analysis - wStance.csv")

# Standardize column names
for df in [ai_low, ai_high]:
    if "text" in df.columns:
        df.rename(columns={"text": "Text"}, inplace=True)

# -------- Calculate embeddings for AI texts --------
model = SentenceTransformer("all-MiniLM-L6-v2")

ai_low["embedding"] = ai_low["Text"].astype(str).apply(lambda x: model.encode(x))
ai_high["embedding"] = ai_high["Text"].astype(str).apply(lambda x: model.encode(x))

# -------- Save results --------
ai_low.to_csv("ai_low_with_embeddings.csv", index=False)
ai_high.to_csv("ai_high_with_embeddings.csv", index=False)

print("✅ Finished computing embeddings. Saved to:")
print(" - ai_low_with_embeddings.csv")
print(" - ai_high_with_embeddings.csv")


## Creating theme file

In [ ]:
# build_comprehensive_theme_file_exact_FIXED.py
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.metrics.pairwise import cosine_similarity
import re

HUMAN_PATH = "all_conditions_with_embeddings.csv"
AI_LOW_PATH = "ai_low_with_embeddings.csv"
AI_HIGH_PATH = "ai_high_with_embeddings.csv"
OUT_PATH = "comprehensive_theme_file.csv"

# ----- EXACT ColumnID mapping (fixed for HIGH: sexism-1 is Q4_Q5, not Q5.1) -----
EXPLICIT_MAP = {
    "baseline": [
        ("Q16","Q18","sexism-1"),
        ("Q4","Q5","sexism-2"),
        ("Q20","Q22","sexism-3"),
        ("Q24","Q26","sexism-4"),
        ("Q8","Q10","aita-1"),
        ("Q28","Q30","aita-2"),
        ("Q32","Q34","aita-3"),
        ("Q36","Q38","aita-4"),
    ],
    "low": [
        ("Q4","Q5","sexism-1"),
        ("Q18","Q20","sexism-2"),
        ("Q24","Q26","sexism-3"),
        ("Q30","Q32","sexism-4"),
        ("Q8","Q10","aita-1"),
        ("Q38","Q40","aita-2"),
        ("Q44.1","Q46","aita-3"),
        ("Q50","Q52","aita-4"),
    ],
    "high": [
        ("Q4","Q5","sexism-1"),        # <-- FIXED
        ("Q2","Q5.1","sexism-2"),
        ("Q2.1","Q5.2","sexism-3"),
        ("Q2.2","Q5.3","sexism-4"),
        ("Q8","Q10","aita-1"),
        ("Q2.3","Q5.4","aita-2"),
        ("Q2.4","Q5.5","aita-3"),
        ("Q2.5","Q5.6","aita-4"),
    ],
}

import re

import re, unicodedata

import re, unicodedata
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load once
_model = SentenceTransformer("all-MiniLM-L6-v2")

def canonical_scenario_label(s: str) -> str:
    """Map 'A1', 'a1', 'aita-1', 'sexism—3', 'sexism 2', etc. → 'aita-1' / 'sexism-3'."""
    if s is None:
        return ""
    t = unicodedata.normalize("NFKC", str(s)).lower().strip()
    t = re.sub(r"[‐-‒–—﹘﹣－]", "-", t)      # all dashes → '-'
    t = re.sub(r"[\s_]+", "", t)              # remove spaces/underscores
    for pat, rep in [(r"^(a|aita)-?([1-4])$", r"aita-\2"),
                     (r"^(s|sexism)-?([1-4])$", r"sexism-\2"),
                     (r"^(aita|sexism)-?([1-4])$", r"\1-\2")]:
        m = re.fullmatch(pat, t)
        if m: return re.sub(pat, rep, t)
    return t

def cos(a, b):
    if a is None or b is None: return np.nan
    a = np.asarray(a).reshape(1, -1); b = np.asarray(b).reshape(1, -1)
    if a.size == 0 or b.size == 0: return np.nan
    return float(cosine_similarity(a, b)[0, 0])

def ensure_text_col(df: pd.DataFrame):
    """Return the name of the text column, normalizing to 'text' in the dataframe."""
    if "text" in df.columns:
        return "text"
    if "Text" in df.columns:
        df.rename(columns={"Text": "text"}, inplace=True)
        return "text"
    if "content" in df.columns:
        df.rename(columns={"content": "text"}, inplace=True)
        return "text"
    raise KeyError("No text/Text/content column found")

def encode_texts(df: pd.DataFrame, text_col="text", batch_size=256):
    """Add/replace df['embedding_vec'] with fresh model encodings from text."""
    # Fill NAs to empty string to avoid errors; keep a mask if you want to set None later
    texts = df[text_col].fillna("").astype(str).tolist()
    vecs = _model.encode(texts, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=False)
    df["embedding_vec"] = [v for v in vecs]  # store as numpy arrays
    return df



def harmonize_low_key(s: str) -> str:
    """
    Normalize low-AI scenario keys to 'aita-<n>' or 'sexism-<n>'.
    Accepts: 'a1', 'A 1', 'aita1', 'AITA-1', ' aita 1 ', 's1', 'S 2', 'sexism3', 'Sexism-4', etc.
    """
    t = str(s).strip().lower()
    # remove spaces/underscores
    t = re.sub(r'[\s_]+', '', t)

    # Already in canonical form?
    m = re.fullmatch(r'(aita|sexism)-([1-4])', t)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    # Forms like 'a1' / 'aita1'
    m = re.fullmatch(r'(a|aita)([1-4])', t)
    if m:
        return f"aita-{m.group(2)}"

    # Forms like 's1' / 'sexism1'
    m = re.fullmatch(r'(s|sexism)([1-4])', t)
    if m:
        return f"sexism-{m.group(2)}"

    # Forms like 'aita- 1' / 'sexism -1' (after removing spaces above, these are caught earlier)
    m = re.fullmatch(r'(aita|sexism)-?([1-4])', t)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    # Last resort: return as-is so diagnostics show the unexpected keys
    return str(s).strip()


def build_role_map():
    m = {}
    for cond, triples in EXPLICIT_MAP.items():
        for q_first, q_final, scen in triples:
            m[(cond, str(q_first).strip().lower())] = (scen, "first")
            m[(cond, str(q_final).strip().lower())] = (scen, "final")
    return m

ROLE_MAP = build_role_map()

def pick(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def parse_embedding(x):
    if pd.isna(x):
        return None
    if isinstance(x, (list, np.ndarray)):
        a = np.asarray(x, dtype=float).ravel()
        return a if a.size else None
    try:
        a = np.asarray(literal_eval(str(x)), dtype=float).ravel()
        return a if a.size else None
    except Exception:
        return None



# ---------- HUMAN: assign scenario/role strictly via mapping ----------
df = pd.read_csv(HUMAN_PATH, low_memory=False)

# ---------- Load human rows (first/final) ---------

# Normalize expected cols
if "ColumnID" not in df.columns:
    colid = next((c for c in ["colID","columnid","final_columnid"] if c in df.columns), None)
    if colid: df.rename(columns={colid: "ColumnID"}, inplace=True)

required = ["condition", "response_id", "scenario", "role"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in human file: {missing}")

# Recompute embeddings from raw text (ignores any serialized vectors)
text_col = ensure_text_col(df)  # normalizes to 'text'
df = encode_texts(df, text_col=text_col)


if "ColumnID" not in df.columns:
    colid = pick(df, ["colID","columnid","final_columnid"])
    if colid: df = df.rename(columns={colid:"ColumnID"})

for c in ["condition","response_id","ColumnID","embedding"]:
    if c not in df.columns:
        raise ValueError(f"Human file missing required column: {c}")

df["embedding_vec"] = df["embedding"].apply(parse_embedding)
df["colid_norm"] = df["ColumnID"].astype(str).str.strip().str.lower()
df[["scenario","role"]] = df.apply(
    lambda r: pd.Series(ROLE_MAP.get((r["condition"], r["colid_norm"]), (np.nan, np.nan))),
    axis=1
)

# ---------- first–final cosine ----------
pairs = []
for (cond, rid, scen), g in df.groupby(["condition","response_id","scenario"], dropna=False):
    if pd.isna(scen): 
        continue
    first = g.loc[g["role"]=="first"]
    final = g.loc[g["role"]=="final"]
    if first.empty or final.empty:
        continue
    r_first, r_final = first.iloc[0], final.iloc[0]
    pairs.append({
        "condition": cond,
        "response_id": rid,
        "scenario": scen,
        "cosine_first_final": cos(r_first["embedding_vec"], r_final["embedding_vec"])
    })
pair_df = pd.DataFrame(pairs)

# ---------- AI: load by scenario (fix LOW 'a1/s1' → 'aita-1/sexism-1') ----------
def harmonize_low_key(s):
    t = str(s).strip().lower()
    m = re.fullmatch(r"a\s*([1-4])", t)
    if m: return f"aita-{int(m.group(1))}"
    m = re.fullmatch(r"s\s*([1-4])", t)
    if m: return f"sexism-{int(m.group(1))}"
    return str(s).strip()  # leave as-is otherwise

def load_ai_embeddings_reencode(path: str, condition: str) -> dict:
    """
    Load AI file, canonicalize scenario keys, and recompute embeddings from raw text.
    Returns: dict[scenario_key] -> np.ndarray
    """
    ai = pd.read_csv(path, low_memory=False)

    # Figure out scenario column in AI file
    scen_col = None
    for c in ["scenario","Scenario","scenario_key_human","scenario_key","scenario_id"]:
        if c in ai.columns:
            scen_col = c; break
    if scen_col is None:
        # if low file uses shorthand columns like 'a1','s1' etc, build a scenario column first if present
        # but most of your AI files already have a scenario column; if not, raise here:
        raise ValueError(f"{path}: couldn't find a scenario column")

    # Ensure text column and re-encode
    tcol = ensure_text_col(ai)  # normalizes to 'text'
    ai = encode_texts(ai, text_col=tcol)

    # Canonical scenario keys
    ai["scenario_key"] = ai[scen_col].apply(canonical_scenario_label)

    # Keep first per scenario_key; drop rows without a usable embedding
    ai = (ai.dropna(subset=["scenario_key"])
            .drop_duplicates(subset=["scenario_key"], keep="first"))

    # Diagnostics
    keys = sorted(ai["scenario_key"].unique().tolist())
    print(f"\n[AI LOAD] {condition.upper()} {path}")
    print("  Scenario keys:", keys)

    # Build dict
    return dict(zip(ai["scenario_key"], ai["embedding_vec"]))




ai_low_map  = load_ai_embeddings_reencode(AI_LOW_PATH,  "low")
ai_high_map = load_ai_embeddings_reencode(AI_HIGH_PATH, "high")


# ---- Clear diagnostics to verify matches ----
hum_low  = set(df.loc[(df["condition"]=="low")  & df["role"].eq("final"), "scenario"].dropna().astype(str))
hum_high = set(df.loc[(df["condition"]=="high") & df["role"].eq("final"), "scenario"].dropna().astype(str))

print("\n[DIAG] Human FINAL scenarios:")
print("LOW :", sorted(hum_low))
print("HIGH:", sorted(hum_high))

print("\n[DIAG] AI scenario keys (normalized):")
print("LOW :", sorted(ai_low_map.keys()))
print("HIGH:", sorted(ai_high_map.keys()))

print("\n[DIAG] Overlap:")
print("LOW  overlap:", sorted(hum_low & set(ai_low_map.keys())))
print("HIGH overlap:", sorted(hum_high & set(ai_high_map.keys())))


# ---------- final–AI cosine ----------
final_ai_rows = []
key_cols = ["condition", "response_id", "scenario"]

for key, g in df.groupby(key_cols, dropna=False):
    cond, rid, scenario = key
    final = g.loc[g["role"].str.lower() == "final"]
    if final.empty:
        final_ai_rows.append({"condition":cond,"response_id":rid,"scenario":scenario,"cosine_final_ai":np.nan})
        continue

    final_vec = final.iloc[0]["embedding_vec"]

    if cond not in {"low","high"}:
        final_ai_rows.append({"condition":cond,"response_id":rid,"scenario":scenario,"cosine_final_ai":np.nan})
        continue

    scen_key = canonical_scenario_label(scenario)
    ai_vec = (ai_low_map if cond == "low" else ai_high_map).get(scen_key)

    # Optional debug:
    # print(f"[{cond}] scenario={scenario!r} -> {scen_key!r} | ai_vec={'ok' if ai_vec is not None else 'NONE'}")

    final_ai_rows.append({
        "condition": cond,
        "response_id": rid,
        "scenario": scenario,
        "cosine_final_ai": cos(final_vec, ai_vec)
    })

final_ai_df = pd.DataFrame(final_ai_rows)


# ---------- Merge ----------
out = (pair_df
       .merge(final_ai_df, on=["condition","response_id","scenario"], how="outer", validate="one_to_one")
       .sort_values(["condition","response_id","scenario"])
       .drop_duplicates(subset=["condition","response_id","scenario"], keep="first")
       .reset_index(drop=True))

out.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")
print("\nCounts by condition (first–final):")
print(out.groupby("condition")["cosine_first_final"].count())
print("\nCounts by condition (final–AI present):")
print(out.groupby("condition")["cosine_final_ai"].apply(lambda s: s.notna().sum()))


## Details file

In [ ]:
# build_comprehensive_detail_lengths.py
import re, unicodedata
import numpy as np
import pandas as pd

# ---------- Paths ----------
HUMAN_PATH   = "all_conditions_with_embeddings.csv"   # has condition/response_id/scenario/role/Text or text
AI_LOW_PATH  = "ai_low_with_embeddings.csv"           # has scenario + Text or text (AI explanations)
AI_HIGH_PATH = "ai_high_with_embeddings.csv"          # has scenario + Text or text (AI explanations)
OUT_PATH     = "comprehensive_detail_lengths.csv"

# ---------- Helpers ----------
def canonical_scenario_label(s: str) -> str:
    """
    Map a variety of scenario labels to canonical 'sexism-1'..'sexism-4' or 'aita-1'..'aita-4'.
    Robust to spaces, underscores, different dashes, and case.
    """
    if s is None:
        return ""
    t = unicodedata.normalize("NFKC", str(s)).lower().strip()
    # normalize dashes and remove spaces/underscores
    t = re.sub(r"[‐-‒–—﹘﹣－-]", "-", t)
    t = re.sub(r"[\s_]+", "", t)
    # common patterns
    m = re.fullmatch(r"(aita|a)-?([1-4])", t)
    if m: return f"aita-{m.group(2)}"
    m = re.fullmatch(r"(sexism|s)-?([1-4])", t)
    if m: return f"sexism-{m.group(2)}"
    # already looks canonical?
    m = re.fullmatch(r"(aita|sexism)-([1-4])", t)
    if m: return t
    return t

def ensure_text_col(df: pd.DataFrame) -> str:
    """Ensure a 'text' column exists; rename Text/content->text if needed; returns column name 'text'."""
    if "text" in df.columns:
        return "text"
    if "Text" in df.columns:
        df.rename(columns={"Text": "text"}, inplace=True)
        return "text"
    if "content" in df.columns:
        df.rename(columns={"content": "text"}, inplace=True)
        return "text"
    raise KeyError("No text/Text/content column found in dataframe.")

def pick(df: pd.DataFrame, candidates):
    """Return the first column from candidates that exists, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def wc_and_cc(s: str):
    """Return (word_count, char_count) for a string.
    If the text is empty/whitespace/NaN → return (np.nan, np.nan)."""
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s_strip = s.strip()
    if not s_strip:   # empty string
        return np.nan, np.nan
    words = len(s_strip.split())
    chars = len(s)
    return words, chars

# ---------- Load human rows (first/final) ----------
df = pd.read_csv(HUMAN_PATH, low_memory=False)

# Normalize expected columns
text_col = ensure_text_col(df)
if "condition" not in df.columns or "response_id" not in df.columns or "scenario" not in df.columns or "role" not in df.columns:
    raise ValueError("Human file must contain columns: condition, response_id, scenario, role, and text/Text/content.")

# Canonical scenario key
df["scenario_key"] = df["scenario"].apply(canonical_scenario_label)

# Keep useful columns and drop exact duplicate keys (keep first)
df_h = (df[["condition", "response_id", "scenario", "scenario_key", "role", "text"]]
          .copy())
df_h["role"] = df_h["role"].astype(str).str.lower().str.strip()
df_h = df_h.drop_duplicates(subset=["condition","response_id","scenario_key","role","text"], keep="first")

# ---------- Prepare AI text maps (low & high) ----------
def load_ai_text_map(path: str, condition: str) -> dict:
    ai = pd.read_csv(path, low_memory=False)
    tcol = ensure_text_col(ai)
    # find scenario column in AI file
    scen_col = pick(ai, ["scenario","Scenario","scenario_key","scenario_key_human","scenario_label", "scenario_id"])
    if scen_col is None:
        raise ValueError(f"{path}: couldn't find a scenario column among "
                         "['scenario','Scenario','scenario_key','scenario_key_human','scenario_label']")
    ai["scenario_key"] = ai[scen_col].apply(canonical_scenario_label)
    # keep first per scenario_key with non-empty text
    ai = (ai.dropna(subset=["scenario_key"])
            .drop_duplicates(subset=["scenario_key"], keep="first"))
    # build map: scenario_key -> AI text
    return dict(zip(ai["scenario_key"], ai[tcol].astype(str)))

ai_low_map  = load_ai_text_map(AI_LOW_PATH,  "low")
ai_high_map = load_ai_text_map(AI_HIGH_PATH, "high")

# ---------- Aggregate to one row per (condition, response_id, scenario) ----------
rows = []
key_cols = ["condition","response_id","scenario_key"]

for (cond, rid, scen_key), g in df_h.groupby(key_cols, dropna=False):
    # pick first/final texts (if multiple, keep first occurrence)
    g_first = g.loc[g["role"]=="first"]
    g_final = g.loc[g["role"]=="final"]
    first_text = g_first.iloc[0]["text"] if not g_first.empty else ""
    final_text = g_final.iloc[0]["text"] if not g_final.empty else ""

    # AI text only for low/high
    if cond == "low":
        ai_text = ai_low_map.get(scen_key, "")
    elif cond == "high":
        ai_text = ai_high_map.get(scen_key, "")
    else:
        ai_text = ""  # baseline → NA later

    # lengths
    first_w, first_c = wc_and_cc(first_text)
    final_w, final_c = wc_and_cc(final_text)
    ai_w,    ai_c    = wc_and_cc(ai_text)

    # deltas
    delta_words_first_final = final_w - first_w
    delta_chars_first_final = final_c - first_c

    if cond in ("low","high") and ai_text != "":
        delta_words_final_ai = ai_w - final_w
        delta_chars_final_ai = ai_c - final_c
    else:
        delta_words_final_ai = np.nan
        delta_chars_final_ai = np.nan

    # original scenario label (for readability) — pick the first seen
    scen_orig = g.iloc[0]["scenario"]

    rows.append({
        "condition": cond,
        "response_id": str(rid),
        "scenario": scen_orig,
        "scenario_key": scen_key,

        "initial_words": first_w,
        "initial_chars": first_c,
        "final_words": final_w,
        "final_chars": final_c,
        "ai_words": ai_w if cond in ("low","high") else np.nan,
        "ai_chars": ai_c if cond in ("low","high") else np.nan,

        "delta_words_first_final": delta_words_first_final,
        "delta_chars_first_final": delta_chars_first_final,
        "delta_words_final_ai": delta_words_final_ai,
        "delta_chars_final_ai": delta_chars_final_ai,
    })

out = pd.DataFrame(rows).sort_values(["condition","response_id","scenario_key"]).reset_index(drop=True)

# ---------- Diagnostics ----------
print("\nCounts by condition:")
print(out.groupby("condition")["scenario_key"].count())

print("\nExample rows:")
print(out.head(8).to_string(index=False))

# ---------- Save ----------
out.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")


## Stance file

In [ ]:
# build_comprehensive_stance_file.py
import pandas as pd
import numpy as np

HUMAN_PATH   = "all_conditions_with_embeddings.csv"
AI_LOW_PATH  = "ai_low_with_embeddings.csv"
AI_HIGH_PATH = "ai_high_with_embeddings.csv"
OUT_PATH     = "comprehensive_stance_file.csv"

def pick(df: pd.DataFrame, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    raise KeyError(f"None of {candidates} found. Available: {list(df.columns)}")

# ---------------- Stance helpers ----------------
def short_form_human_stance(s: str):
    if pd.isna(s):
        return np.nan
    t = str(s).strip().lower()
    if ("it depends" in t or "depends" in t or "unsure" in t
        or "not sure" in t or "uncertain" in t or "maybe" in t):
        return "it depends"
    if t.startswith("yes") or (" sexist" in t and "not" not in t):
        return "yes"
    if t.startswith("no") or "not sexist" in t:
        return "no"
    return "it depends"

def normalize_ai_stance(x: str):
    if pd.isna(x):
        return np.nan
    t = str(x).strip().lower()
    if t in ("yes","y","true","1"):
        return "yes"
    if t in ("no","n","false","0"):
        return "no"
    return np.nan

# ---------------- Loaders ----------------
def load_human(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    # columns (case-insensitive)
    stance_col = pick(df, ["stance"])
    cond_col   = pick(df, ["condition"])
    resp_col   = pick(df, ["response_id"])
    scen_col   = pick(df, ["scenario"])
    role_col   = pick(df, ["role"])

    # keep just what we need
    df = df[[cond_col, resp_col, scen_col, role_col, stance_col]].copy()
    df.columns = ["condition","response_id","scenario","role","stance"]

    # normalize BEFORE anything else to prevent a first-only row splitting from final-only row
    df["condition"]   = df["condition"].astype(str).str.strip().str.lower()
    df["response_id"] = df["response_id"].astype(str).str.strip()
    df["scenario"]    = df["scenario"].astype(str).str.strip().str.lower()
    df["role"]        = df["role"].astype(str).str.strip().str.lower()

    # drop exact dups and keep first row per (cond, id, scen, role)
    before = len(df)
    df = df.drop_duplicates(subset=["condition","response_id","scenario","role"], keep="first")
    after = len(df)
    if before != after:
        print(f"[HUMAN] dropped {before-after} duplicate rows on (condition,response_id,scenario,role).")

    return df

def harmonize_low_scenario(s):
    s = str(s).strip().lower()
    # s1 → sexism-1 ; a3 → aita-3 ; also accept s1.0 / a2.0 etc.
    if s.startswith("s") and s[1:].replace(".","").isdigit():
        return f"sexism-{int(float(s[1:]))}"
    if s.startswith("a") and s[1:].replace(".","").isdigit():
        return f"aita-{int(float(s[1:]))}"
    return s

def load_ai(path: str, condition: str) -> pd.DataFrame:
    ai = pd.read_csv(path, low_memory=False)
    if condition == "low":
        scen_col = pick(ai, ["Scenario"])
        ai["_scenario"] = ai[scen_col].apply(harmonize_low_scenario)
    else:
        scen_col = pick(ai, ["scenario_id"])
        ai["_scenario"] = ai[scen_col].astype(str).str.strip().str.lower()

    stance_col = pick(ai, ["stance","Stance"])
    ai["_stance_norm"] = ai[stance_col].apply(normalize_ai_stance)

    # One AI stance per scenario
    ai = (ai.dropna(subset=["_scenario"])
            .drop_duplicates(subset=["_scenario"], keep="first"))
    return (ai[["_scenario","_stance_norm"]]
            .rename(columns={"_scenario":"scenario","_stance_norm":"ai_stance_norm"})
            .assign(condition=condition))

# ---------------- Main ----------------
def main():
    human = load_human(HUMAN_PATH)

    # DIAGNOSTIC: count scenarios per respondent/condition BEFORE pivot
    counts = (human.groupby(["condition","response_id"])["scenario"]
                    .nunique()
                    .rename("n_scenarios")
                    .reset_index())
    too_many = counts[counts["n_scenarios"] > 8]
    if not too_many.empty:
        print("\n[DIAG] Respondents with >8 scenarios (possible mapping/dup issue):")
        print(too_many.head(20).to_string(index=False))

    too_many4 = counts[counts["n_scenarios"] > 4]
    if not too_many4.empty:
        print("\n[DIAG] Respondents with >4 scenarios (if you expected 4 per cond):")
        print(too_many4.head(20).to_string(index=False))

    # Keep first/final only (after normalization and dedup)
    human2 = human[human["role"].isin(["first","final"])].copy()

    # Pivot to wide — one row per (condition, response_id, scenario)
    wide = (human2.pivot_table(
                index=["condition","response_id","scenario"],
                columns="role",
                values="stance",
                aggfunc="first")
            .reset_index()
            .rename(columns={"first":"stance_first","final":"stance_final"}))

    # DIAGNOSTIC: rows missing first or final
    missing_first = wide["stance_first"].isna().sum()
    missing_final = wide["stance_final"].isna().sum()
    print(f"\n[DIAG] Missing first: {missing_first} | Missing final: {missing_final}")

    # flip check
    wide["flip_first_final"] = (
        wide["stance_first"].notna()
        & wide["stance_final"].notna()
        & (wide["stance_first"] != wide["stance_final"])
    )

    # short form of final
    wide["final_stance_short"] = wide["stance_final"].apply(short_form_human_stance)

    # Load AI stance (normalized)
    ai_low  = load_ai(AI_LOW_PATH,  "low")
    ai_high = load_ai(AI_HIGH_PATH, "high")

    # Merge by (condition, scenario) — condition already normalized to 'low'/'high'
    wide = wide.merge(ai_low,  on=["condition","scenario"], how="left", validate="m:1")
    wide = wide.merge(ai_high.rename(columns={"ai_stance_norm":"ai_stance_norm_high"}),
                      on=["condition","scenario"], how="left", validate="m:1")

    # Select AI stance for the row's condition
    wide["ai_stance_final_norm"] = np.where(
        wide["condition"].eq("low"),  wide["ai_stance_norm"],
        np.where(wide["condition"].eq("high"), wide["ai_stance_norm_high"], np.nan)
    )
    wide = wide.drop(columns=["ai_stance_norm","ai_stance_norm_high"], errors="ignore")

    # Compare (only for low/high, flipped rows, with usable yes/no on both sides)
    evaluable = (
        wide["condition"].isin(["low","high"])
        & wide["flip_first_final"]
        & wide["final_stance_short"].isin(["yes","no"])
        & wide["ai_stance_final_norm"].isin(["yes","no"])
    )
    wide["stance_ai_flip"] = np.where(
        evaluable,
        (wide["final_stance_short"] == wide["ai_stance_final_norm"]),
        np.nan
    )

    # FINAL de-dup (paranoia): one row per (cond, id, scen)
    wide = (wide.sort_values(["condition","response_id","scenario"])
                 .drop_duplicates(subset=["condition","response_id","scenario"], keep="first")
                 .reset_index(drop=True))

    # DIAGNOSTIC: count rows per (condition,response_id)
    n_per_id = (wide.groupby(["condition","response_id"])["scenario"]
                    .nunique()
                    .rename("n_rows"))
    offenders = n_per_id[n_per_id > 8]
    if not offenders.empty:
        print("\n[DIAG] After pivot: respondents with >8 rows per condition:")
        print(offenders.head(20).to_string())

    # Save
    cols_out = [
        "condition","response_id","scenario",
        "stance_first","stance_final","final_stance_short",
        "flip_first_final","ai_stance_final_norm","stance_ai_flip"
    ]
    wide[cols_out].to_csv(OUT_PATH, index=False)
    print(f"\n✅ Saved: {OUT_PATH}")

    # Quick counts you may expect (per condition)
    print("\n[Counts per condition]")
    print(wide.groupby("condition")["scenario"].count())
    print("\n[Flips per condition]")
    print(wide.groupby("condition")["flip_first_final"].sum(min_count=1))

if __name__ == "__main__":
    main()


## Language file

In [ ]:
# build_comprehensive_language_file.py
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ---------- Paths ----------
HUMAN_PATH   = "LIWC-22 Results - all_conditions_with_embeddings - LIWC Analysis.csv"
AI_LOW_PATH  = "LIWC-22 Results - low_interactivity_AI_explanati___ - LIWC Analysis- wStance.csv"
AI_HIGH_PATH = "LIWC-22 Results - AIhigh - LIWC Analysis - wStance.csv"
OUT_PATH     = "comprehensive_language_file.csv"

# ---------- Utilities ----------
def pick(df: pd.DataFrame, candidates):
    """Pick the first existing column (case-sensitive) from candidates."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def liwc_span(df: pd.DataFrame, start="WC", end="Emoji") -> list:
    """Return an ordered list of LIWC feature columns from start..end inclusive."""
    cols = list(df.columns)
    try:
        i0 = cols.index(start)
    except ValueError:
        raise KeyError(f"Start LIWC column '{start}' not found.")
    try:
        i1 = cols.index(end)
    except ValueError:
        raise KeyError(f"End LIWC column '{end}' not found.")
    if i1 < i0:
        raise ValueError(f"LIWC end '{end}' appears before start '{start}'.")
    return cols[i0:i1+1]

def to_vec(row: pd.Series, liwc_cols: list) -> np.ndarray:
    """Row → 1D float vector (NaNs → 0)."""
    v = pd.to_numeric(row[liwc_cols], errors="coerce").to_numpy(dtype=float)
    # Safe: replace NaNs with 0s so cosine can run (norm checks below)
    v = np.nan_to_num(v, nan=0.0)
    return v

def cos(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity with zero-norm guard."""
    if a is None or b is None:
        return np.nan
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float((a @ b) / (na * nb))

def harmonize_low_key(s: str) -> str:
    """Map 's1'→'sexism-1', 'a3'→'aita-3' for low AI file."""
    s = str(s).strip().lower()
    if len(s) >= 2 and s[0] in ("s", "a") and s[1:].replace(".", "").isdigit():
        idx = int(float(s[1:]))  # handles '1' and '1.0'
        if s[0] == "s":
            return f"sexism-{idx}"
        else:
            return f"aita-{idx}"
    return s

# ---------- Load Human (all conditions) ----------
human = pd.read_csv(HUMAN_PATH, low_memory=False)

# Required core columns
need = ["condition", "response_id", "scenario", "role"]
missing = [c for c in need if c not in human.columns]
if missing:
    raise ValueError(f"[HUMAN] Missing required columns: {missing}")

# Normalize identifiers
human["condition"]   = human["condition"].astype(str).str.strip().str.lower()
human["response_id"] = human["response_id"].astype(str).str.strip()
human["scenario"]    = human["scenario"].astype(str).str.strip().str.lower()
human["role"]        = human["role"].astype(str).str.strip().str.lower()

# Identify LIWC span and ensure numeric
liwc_cols = liwc_span(human, start="WC", end="Emoji")
human[liwc_cols] = human[liwc_cols].apply(pd.to_numeric, errors="coerce")

# Keep only (first/final) rows and drop duplicate keys, keeping first
human_ff = (
    human[human["role"].isin(["first", "final"])]
    .drop_duplicates(subset=["condition", "response_id", "scenario", "role"], keep="first")
    .copy()
)

# ---------- Compute cosine_first_final per (cond, id, scenario) ----------
pairs = []
for (cond, rid, scen), g in human_ff.groupby(["condition", "response_id", "scenario"], dropna=False):
    g_first = g[g["role"] == "first"]
    g_final = g[g["role"] == "final"]
    if g_first.empty or g_final.empty:
        continue
    v_first = to_vec(g_first.iloc[0], liwc_cols)
    v_final = to_vec(g_final.iloc[0], liwc_cols)
    pairs.append({
        "condition": cond,
        "response_id": rid,
        "scenario": scen,
        "cosine_first_final_liwc": cos(v_first, v_final),
        # store final vector for later AI match to avoid re-reading
        "_final_vec": v_final
    })
pair_df = pd.DataFrame(pairs)

# ---------- Load AI LIWC and build scenario→vector maps ----------
def load_ai_liwc(path: str, condition: str) -> dict:
    ai = pd.read_csv(path, low_memory=False)

    # figure out scenario column name by condition
    if condition == "low":
        scen_col = pick(ai, ["Scenario"])  # low AI file uses 'Scenario' like s1 / a3
        if not scen_col:
            raise ValueError("[AI low] Could not find 'Scenario' column.")
        ai["scenario_key"] = ai[scen_col].apply(harmonize_low_key)
    else:
        scen_col = pick(ai, ["scenario_id", "Scenario", "scenario"])
        if not scen_col:
            raise ValueError("[AI high] Could not find a scenario column (scenario_id/Scenario).")
        ai["scenario_key"] = ai[scen_col].astype(str).str.strip().str.lower()

    # LIWC features
    liwc_ai_cols = liwc_span(ai, start="WC", end="Emoji")
    ai[liwc_ai_cols] = ai[liwc_ai_cols].apply(pd.to_numeric, errors="coerce")

    # build vector per scenario (first occurrence wins)
    ai = ai.dropna(subset=["scenario_key"]).drop_duplicates(subset=["scenario_key"], keep="first")
    scen2vec = {}
    for _, r in ai.iterrows():
        vec = np.nan_to_num(r[liwc_ai_cols].to_numpy(dtype=float), nan=0.0)
        scen2vec[r["scenario_key"]] = vec
    return scen2vec

ai_low_map  = load_ai_liwc(AI_LOW_PATH,  "low")
ai_high_map = load_ai_liwc(AI_HIGH_PATH, "high")

# ---------- Compute cosine_final_ai_liwc ----------
def ai_vec_for(cond: str, scenario: str) -> np.ndarray | None:
    if cond == "low":
        return ai_low_map.get(scenario)
    if cond == "high":
        return ai_high_map.get(scenario)
    return None  # baseline has no AI

final_ai_rows = []
miss_debug_rows = []  # collect misses to help debugging
for _, r in pair_df.iterrows():
    cond, rid, scen = r["condition"], r["response_id"], r["scenario"]
    final_vec = r["_final_vec"]
    if cond not in {"low", "high"}:
        final_ai_rows.append({"condition": cond, "response_id": rid, "scenario": scen, "cosine_final_ai_liwc": np.nan})
        continue

    ai_vec = ai_vec_for(cond, scen)
    if ai_vec is None:
        miss_debug_rows.append((cond, scen))
        cos_val = np.nan
    else:
        cos_val = cos(final_vec, ai_vec)

    final_ai_rows.append({
        "condition": cond,
        "response_id": rid,
        "scenario": scen,
        "cosine_final_ai_liwc": cos_val
    })

final_ai_df = pd.DataFrame(final_ai_rows)

# ---------- Merge & clean ----------
out = (
    pair_df.drop(columns=["_final_vec"])
           .merge(final_ai_df, on=["condition","response_id","scenario"], how="left", validate="one_to_one")
           .sort_values(["condition","response_id","scenario"])
           .reset_index(drop=True)
)

# ---------- Diagnostics ----------
print("\nCounts by condition (first–final LIWC cosine):")
print(out.groupby("condition")["cosine_first_final_liwc"].count())

print("\nCounts by condition (final–AI LIWC cosine present):")
print(out.groupby("condition")["cosine_final_ai_liwc"].apply(lambda s: s.notna().sum()))

if miss_debug_rows:
    miss_df = pd.DataFrame(miss_debug_rows, columns=["condition","scenario"]).value_counts().reset_index(name="n")
    print("\n⚠️ Human final rows with NO matching AI scenario vector:")
    print(miss_df.sort_values(["condition","scenario"]).to_string(index=False))

# ---------- Save ----------
out.to_csv(OUT_PATH, index=False)
print(f"\n✅ Saved: {OUT_PATH}")


## Analysis

## Theme

In [ ]:
# stats_comprehensive_theme.py
import pandas as pd
import numpy as np
from itertools import combinations
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

# ---------- Load & clean ----------
file_path = "comprehensive_theme_file.csv"
df = pd.read_csv(file_path)

# Keep one row per (response_id, scenario)
df_clean = (
    df.sort_values(["response_id", "scenario"])
      .drop_duplicates(subset=["response_id", "scenario"], keep="first")
      .reset_index(drop=True)
)

print(f"Rows after clean: {len(df_clean)}")
print(f"Unique response_ids: {df_clean['response_id'].nunique()}")

# ---------- Helper functions ----------
def cohen_d_independent(a, b):
    na, nb = len(a), len(b)
    sa2, sb2 = np.var(a, ddof=1), np.var(b, ddof=1)
    s_pooled = np.sqrt(((na-1)*sa2 + (nb-1)*sb2) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / s_pooled

def pairwise_welch_bh(df, value_col, group_col, alpha=0.05):
    """Run Welch's t-tests pairwise + BH correction."""
    groups = df[group_col].dropna().unique()
    pairs = list(combinations(groups, 2))
    results, pvals = [], []

    for g1, g2 in pairs:
        a = df.loc[df[group_col] == g1, value_col].dropna().values
        b = df.loc[df[group_col] == g2, value_col].dropna().values
        if len(a) < 2 or len(b) < 2:
            continue
        t, p = stats.ttest_ind(a, b, equal_var=False)  # Welch
        d = cohen_d_independent(a, b)
        results.append({
            "pair": f"{g1} vs {g2}",
            "t": t, "p_raw": p,
            "cohens_d": d,
            "n1": len(a), "n2": len(b)
        })
        pvals.append(p)

    # BH correction
    rej, p_bh, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")
    for r, p_adj, reject in zip(results, p_bh, rej):
        r["p_bh"] = p_adj
        r["reject_H0"] = reject

    return pd.DataFrame(results).sort_values("p_raw")

# ---------- Analysis 1: first–final cosine ----------
print("\n=== First–Final Cosine Similarity ===")
df_ff = df_clean[["condition","cosine_first_final"]].dropna()

# ANOVA
anova_model = ols("cosine_first_final ~ C(condition)", data=df_ff).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)
print("ANOVA:\n", anova_table, "\n")

# Group means
means = df_ff.groupby("condition")["cosine_first_final"].agg(["mean","std","count"])
print("Group means:\n", means, "\n")

# Pairwise Welch + BH
pw_ff = pairwise_welch_bh(df_ff, "cosine_first_final", "condition")
print("Pairwise Welch tests (BH corrected):")
print(pw_ff.to_string(index=False))

# ---------- Analysis 2: final–AI cosine (low vs high only) ----------
print("\n=== Final–AI Cosine Similarity (low vs high) ===")
df_ai = df_clean[df_clean["condition"].isin(["low","high"])].dropna(subset=["cosine_final_ai"])

print("Group sizes:\n", df_ai.groupby("condition")["cosine_final_ai"].count(), "\n")
desc = df_ai.groupby("condition")["cosine_final_ai"].agg(["mean","std","count"])
print("Descriptives:\n", desc, "\n")

low_vals  = df_ai.loc[df_ai["condition"]=="low","cosine_final_ai"]
high_vals = df_ai.loc[df_ai["condition"]=="high","cosine_final_ai"]

t, p = stats.ttest_ind(low_vals, high_vals, equal_var=False)
d = cohen_d_independent(low_vals, high_vals)
print(f"Welch's t-test: t={t:.3f}, p={p:.5f}, d={d:.3f}")

# ---------- Analysis 3: within-condition comparison (first–final vs final–AI) ----------
print("\n=== Within-condition comparisons (first–final vs final–AI) ===")
df_both = (
    df_clean[df_clean["condition"].isin(["low","high"])]
      .dropna(subset=["cosine_first_final","cosine_final_ai"])
      .copy()
)
df_both["delta"] = df_both["cosine_first_final"] - df_both["cosine_final_ai"]

for cond in ["low","high"]:
    sub = df_both[df_both["condition"]==cond]
    x, y = sub["cosine_first_final"], sub["cosine_final_ai"]
    t, p = stats.ttest_rel(x, y, nan_policy="omit")
    diff = x - y
    dz = np.mean(diff) / np.std(diff, ddof=1)
    print(f"[{cond}] Paired t-test: t={t:.3f}, p={p:.5f}, dz={dz:.3f}")

# Between-condition on Δ
low_d, high_d = df_both[df_both["condition"]=="low"]["delta"], df_both[df_both["condition"]=="high"]["delta"]
t, p = stats.ttest_ind(low_d, high_d, equal_var=False)
d = cohen_d_independent(low_d, high_d)
print("\nΔ between conditions:")
print(f"  mean Δ (low)={low_d.mean():.3f}, mean Δ (high)={high_d.mean():.3f}")
print(f"  Welch's t={t:.3f}, p={p:.5f}, d={d:.3f}")


In [ ]:
# build_comprehensive_with_reliance.py
import pandas as pd
import numpy as np
from scipy import stats

COMP_PATH = "comprehensive_theme_file.csv"   # <-- your comprehensive file (or stance/theme, whichever DV you want)
LOW_PATH  = "LIWC-22 Results - H-AI_Subjectivity_Study_Low_In___ - LIWC Analysis.csv"
HIGH_PATH = "LIWC-22 Results - H-AI_Subjectivity_Study_High_I___ - LIWC Analysis (1).csv"
OUT_PATH  = "comprehensive_with_reliance.csv"

# ---------------- Reliance mapping ----------------
reliance_map = {
    "Not at all": 1,
    "Too little": 2,
    "The right amount": 3,
    "Too much": 4,
    "Completely": 5
}

mapping_low = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q88_1","Q49.1"),
    "sexism-3": ("Q89_1","Q49.2"),
    "sexism-4": ("Q90_1","Q49.3"),
    "aita-1":   ("Q91_1","Q49.4"),
    "aita-2":   ("Q92_1","Q49.5"),
    "aita-3":   ("Q93_1","Q49.6"),
    "aita-4":   ("Q94_1","Q49.8")
}

mapping_high = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q69_1","Q70"),
    "sexism-3": ("Q71_1","Q72"),
    "sexism-4": ("Q73_1","Q74"),
    "aita-1":   ("Q75_1","Q76"),
    "aita-2":   ("Q77_1","Q78"),
    "aita-3":   ("Q79_1","Q80"),
    "aita-4":   ("Q81_1","Q82")
}

def build_reliance_table(path, cond, mapping):
    df = pd.read_csv(path, low_memory=False)
    id_col = "ResponseId" if "ResponseId" in df.columns else "responseid"
    out = []
    for scen,(col1,col2) in mapping.items():
        for rid,row in df.iterrows():
            ridval = str(row[id_col]).strip()
            for col in [col1,col2]:
                if col not in df.columns: 
                    continue
                val = row[col]
                if pd.isna(val): 
                    continue
                out.append({
                    "condition": cond,
                    "response_id": ridval,
                    "scenario": scen,
                    "column_id": col,
                    "reliance_measure": val,
                    "reliance_measure_num": reliance_map.get(val, np.nan)
                })
    return pd.DataFrame(out)



def corr_report(df, condition):
    print(f"\n=== Condition: {condition} ===")
    subset = df[df["condition"] == condition].copy()

    # Coerce the 1–100 slider into numeric (keeps NaN if not present)
    subset["reliance_measure_0100"] = pd.to_numeric(subset.get("reliance_measure"), errors="coerce")

    pairs = {
        "AI-final vs reliance (1–5 Likert)": ("cosine_final_ai", "reliance_measure_num"),
        "AI-final vs reliance (0–100 slider)": ("cosine_final_ai", "reliance_measure_0100"),
    }

    for label, (xcol, ycol) in pairs.items():
        data = subset[[xcol, ycol]].dropna()
        n = len(data)
        if n < 3:
            print(f"{label}: not enough data (n={n})")
            continue
        r_p, p_p = stats.pearsonr(data[xcol], data[ycol])
        r_s, p_s = stats.spearmanr(data[xcol], data[ycol])
        print(f"{label} (n={n}):")
        print(f"  Pearson r = {r_p:.3f}, p = {p_p:.5f}")
        print(f"  Spearman ρ = {r_s:.3f}, p = {p_s:.5f}")



def main():
    # Build reliance tables
    rel_low  = build_reliance_table(LOW_PATH,  "low",  mapping_low)
    rel_high = build_reliance_table(HIGH_PATH, "high", mapping_high)
    rel = pd.concat([rel_low, rel_high], ignore_index=True)
    print("Reliance table shape (before dedupe):", rel.shape)

    rel = rel.drop_duplicates(subset=["condition","response_id","scenario","column_id"])
    print("Reliance table shape (after dedupe):", rel.shape)

    # Load comprehensive
    comp = pd.read_csv(COMP_PATH, low_memory=False)
    comp["response_id"] = comp["response_id"].astype(str).str.strip()
    comp["condition"]   = comp["condition"].astype(str).str.strip().str.lower()
    comp["scenario"]    = comp["scenario"].astype(str).str.strip().str.lower()

    # Merge
    merged = comp.merge(
        rel,
        on=["condition","response_id","scenario"],
        how="left"
    )
    print("Merged shape:", merged.shape)

    # Save
    merged.to_csv(OUT_PATH, index=False)
    print("✅ Saved:", OUT_PATH)

    # Run correlations
    # --- Run correlations ---
    df_sub = merged[merged["condition"].isin(["low","high"])].dropna(subset=["cosine_final_ai"])
    corr_report(df_sub, "low")
    corr_report(df_sub, "high")

if __name__=="__main__":
    main()


## Stance

In [ ]:
# analyze_comprehensive_stance.py
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import chi2_contingency, fisher_exact

# ------------------------------
# Config
# ------------------------------
STANCE_PATH = "comprehensive_stance_file.csv"   # <- your attached file

# ------------------------------
# Helpers
# ------------------------------
def pick(df: pd.DataFrame, candidates):
    """Pick a column name case-insensitively."""
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        lc = c.lower()
        if lc in lower_map:
            return lower_map[lc]
    raise KeyError(f"None of {candidates} found. Available: {list(df.columns)}")

TRUE_SET  = {"true","t","1","yes","y","toward","towards"}
FALSE_SET = {"false","f","0","no","n","away"}

def coerce_bool_preserve(series: pd.Series) -> pd.Series:
    """
    Build a *new* boolean/NaN series from a mixed-type input, without
    destroying the original values in `series`.
    """
    out = pd.Series(index=series.index, dtype="float")

    # bools
    is_bool = series.apply(lambda x: isinstance(x, (bool, np.bool_)))
    out.loc[is_bool] = series[is_bool].astype(bool)

    # numeric 0/1
    as_num = pd.to_numeric(series, errors="coerce")
    mask_num = as_num.isin([0, 1]) & ~series.isna()
    out.loc[mask_num] = as_num[mask_num].astype(bool)

    # strings like "TRUE", "False", " yes "
    s = series.astype(str).str.strip()
    s_lower = s.str.lower()

    out.loc[s_lower.isin(TRUE_SET)]  = True
    out.loc[s_lower.isin(FALSE_SET)] = False

    # real missing -> NaN
    out.loc[series.isna() | (s.str.len() == 0)] = np.nan

    return out.astype("object")  # True/False/NaN

def holm_correction(pvals):
    m = len(pvals)
    order = np.argsort(pvals)
    holm = [None]*m
    for rank, idx in enumerate(order):
        holm_p = (m - rank) * pvals[idx]
        holm[idx] = min(1.0, holm_p)
    # enforce monotone non-decreasing
    running_max = 0.0
    for rank in reversed(range(m)):
        idx = order[rank]
        running_max = max(running_max, holm[idx])
        holm[idx] = running_max
    return holm

def benjamini_hochberg(pvals):
    """BH/FDR (independent/PRDS). Returns adjusted q-values in original order."""
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = np.array(pvals)[order]
    q = np.empty(m, dtype=float)
    prev = 1.0
    for i in range(m-1, -1, -1):
        q_i = ranked[i] * m / (i+1)
        prev = min(prev, q_i)
        q[i] = min(1.0, prev)
    # restore original order
    out = np.empty(m, dtype=float)
    out[order] = q
    return out.tolist()

# ------------------------------
# Load & normalize columns
# ------------------------------
df = pd.read_csv(STANCE_PATH, low_memory=False)

# Required columns (case-insensitive): condition, response_id, scenario,
# stance_first, stance_final, flip_first_final, ai_stance_final, stance_ai_flip
cond_col            = pick(df, ["condition"])
resp_col            = pick(df, ["response_id"])
scen_col            = pick(df, ["scenario"])
stance_first_col    = pick(df, ["stance_first"])
stance_final_col    = pick(df, ["stance_final"])
flip_col            = pick(df, ["flip_first_final"])
ai_stance_final_col = pick(df, ["ai_stance_final_norm"])
stance_ai_col       = pick(df, ["stance_ai_flip"])

# Optional short form column (if present from earlier step)
stance_final_shortcol = None
try:
    stance_final_shortcol = pick(df, ["final_stance_short", "stance_final_short"])
except KeyError:
    stance_final_shortcol = None

# Normalize a few key text columns for safety
for c in [cond_col, resp_col, scen_col]:
    df[c] = df[c].astype(str).str.strip()

# ------------------------------
# Coerce booleans into new columns (preserve originals)
# ------------------------------
df["_orig_flip_first_final"] = df[flip_col]
df["_orig_stance_ai_flip"]   = df[stance_ai_col]

df["flip_first_final_bool"] = coerce_bool_preserve(df[flip_col])
df["stance_ai_flip_bool"]   = coerce_bool_preserve(df[stance_ai_col])

print("\n[AFTER COERCION] value counts")
print("flip_first_final_bool:")
print(df["flip_first_final_bool"].value_counts(dropna=False))
print("\nstance_ai_flip_bool:")
print(df["stance_ai_flip_bool"].value_counts(dropna=False))

# Show unmapped strings (non-empty that didn't coerce)
def show_unmapped(label, original, coerced):
    unmapped = coerced.isna() & original.notna() & original.astype(str).str.strip().ne("")
    if unmapped.any():
        print(f"\n[UNMAPPED → review] {label} (first 25 unique values)")
        print(original.loc[unmapped].astype(str).str.strip().value_counts().head(25))

show_unmapped("flip_first_final", df["_orig_flip_first_final"], df["flip_first_final_bool"])
show_unmapped("stance_ai_flip",   df["_orig_stance_ai_flip"],   df["stance_ai_flip_bool"])

# ------------------------------
# Focused diagnostic: rows that SHOULD have stance_ai_flip
# (non-baseline & flipped)
# ------------------------------
mask_should_have = df[cond_col].str.lower().ne("baseline") & (df["flip_first_final_bool"] == True)
cols_to_show = [
    cond_col, resp_col, scen_col,
    flip_col, "flip_first_final_bool",
    stance_final_col, ai_stance_final_col,
    stance_ai_col, "stance_ai_flip_bool"
]
print("\n[Rows that should have stance_ai_flip] (first 25)")
print(df.loc[mask_should_have, cols_to_show].head(25).to_string(index=False))

# ------------------------------
# Backfill stance_ai_flip_bool when possible
# (only for non-baseline & flipped rows that are NaN but stances available)
# ------------------------------
needs = (
    mask_should_have
    & df["stance_ai_flip_bool"].isna()
    & df[ai_stance_final_col].notna()
)

if stance_final_shortcol:
    lhs_series = df[stance_final_shortcol]
else:
    lhs_series = df[stance_final_col]

def _norm(s): return str(s).strip().lower()

recomputable = needs & lhs_series.notna()
if recomputable.any():
    lhs = lhs_series.loc[recomputable].map(_norm)
    rhs = df.loc[recomputable, ai_stance_final_col].map(_norm)
    df.loc[recomputable, "stance_ai_flip_bool"] = (lhs == rhs)
    print(f"\nBackfilled stance_ai_flip_bool for {recomputable.sum()} rows using "
          f"{stance_final_shortcol or stance_final_col} vs {ai_stance_final_col}.")

# ------------------------------
# De-dup: one row per (response_id, scenario)
# (keep first; this mirrors your prior cleaning approach)
# ------------------------------
df = (
    df.sort_values([resp_col, scen_col])
      .drop_duplicates(subset=[resp_col, scen_col], keep="first")
      .reset_index(drop=True)
)

print("\nRows after de-dup:", len(df))
print("Unique response_ids:", df[resp_col].nunique())
print(df[cond_col].value_counts())

# -----------------------------------------------------------------------------
# ANALYSES
# -----------------------------------------------------------------------------

# ================
# 1) Trial-level: stance change vs no-change by condition
# ================
ct = pd.crosstab(df[cond_col], df["flip_first_final_bool"]).reindex(
    columns=[False, True], fill_value=0
)
chi2, p_chi, dof, exp = chi2_contingency(ct)
print("\n[Trial-level] change vs no-change by condition")
print(ct.rename(columns={False: "no_change", True: "change"}))
print(f"chi2={chi2:.3f}, df={dof}, p={p_chi:.5f}")

# Pairwise condition comparisons on change/no-change with Holm + BH
conds = sorted(df[cond_col].dropna().unique().tolist())
pairwise = []
for c1, c2 in combinations(conds, 2):
    sub = df[df[cond_col].isin([c1, c2])]
    tab = pd.crosstab(sub[cond_col], sub["flip_first_final_bool"]).reindex(
        index=[c1, c2], columns=[False, True], fill_value=0
    )
    chi2, p_raw, dof, exp = chi2_contingency(tab)
    # if any expected cell < 5, use Fisher’s exact (2x2)
    if (exp < 5).any():
        _, p_raw = fisher_exact(tab.values)
        test = "fisher_exact"
    else:
        test = "chi2"
    pairwise.append({"pair": f"{c1} vs {c2}", "test": test, "p_raw": p_raw})

if pairwise:
    pvals = [r["p_raw"] for r in pairwise]
    p_holm = holm_correction(pvals)
    p_bh   = benjamini_hochberg(pvals)
    for i, r in enumerate(pairwise):
        r["p_holm"] = p_holm[i]
        r["p_bh"]   = p_bh[i]
    res_pairwise = pd.DataFrame(pairwise).sort_values("p_raw").reset_index(drop=True)
    print("\nPairwise stance change vs no-change (Holm & BH adjusted):")
    print(res_pairwise.to_string(index=False))
else:
    print("\n(No pairwise comparisons possible.)")

# ================
# 2) Among switchers in LOW vs HIGH: toward-AI vs away-from-AI
# ================
df_lh = df[df[cond_col].str.lower().isin(["low", "high"])].copy()

# “Switchers” are rows with flip_first_final_bool == True
switchers = df_lh[df_lh["flip_first_final_bool"] == True].copy()

# Define toward_ai from stance_ai_flip_bool (True=toward, False=away)
# If still NaN after backfill, they will be dropped before the test.
switchers = switchers.dropna(subset=["stance_ai_flip_bool"])
switchers["toward_ai"] = switchers["stance_ai_flip_bool"].astype(bool)

if switchers.empty:
    print("\nNo evaluable flips in low/high to compare (toward vs away).")
else:
    ct_toward = pd.crosstab(
        switchers[cond_col], switchers["toward_ai"]
    ).reindex(index=["low","high"], columns=[False, True], fill_value=0)

    chi2_t, p_t, dof_t, exp_t = chi2_contingency(ct_toward)
    # If expected < 5, fall back to Fisher (2x2)
    if (exp_t < 5).any():
        _, p_t = fisher_exact(ct_toward.values)
        print( _, p_t )
        used = "fisher_exact"
    else:
        used = "chi2"

    print("\n[Toward vs Away] among switchers, by condition (low vs high)")
    print(ct_toward.rename(columns={False: "away_from_AI", True: "toward_AI"}))
    print(f"{used}: chisq = {_:.5f} p={p_t:.5f}")
    print(fisher_exact(ct_toward.values))

# ------------------------------
# Counts: flips and flips toward AI (by condition)
# ------------------------------

print("\n=== Flip counts by condition (all rows) ===")
# total rows and total flips per condition
by_cond = (
    df.groupby(cond_col)
      .agg(
          n_rows=('flip_first_final_bool', 'size'),
          n_flips=('flip_first_final_bool', lambda s: s.fillna(False).sum())
      )
      .reset_index()
)
by_cond['flip_rate'] = by_cond['n_flips'] / by_cond['n_rows']
print(by_cond.to_string(index=False))

print("\n=== Flip directions among switchers (low/high) ===")
# Among low/high switchers with evaluable AI direction
if switchers.empty:
    print("No evaluable switchers in low/high.")
else:
    # Count direction per condition
    dir_counts = (
        switchers
        .groupby(cond_col)['toward_ai']
        .value_counts(dropna=False)
        .unstack(fill_value=0)
        .rename(columns={False: 'away_from_AI', True: 'toward_AI'})
        .reindex(columns=['away_from_AI', 'toward_AI'], fill_value=0)
        .reset_index()
    )

    # Add totals and rates
    dir_counts['n_switchers_evaluable'] = dir_counts['away_from_AI'] + dir_counts['toward_AI']
    # Avoid div-by-zero
    for col in ['away_from_AI', 'toward_AI']:
        dir_counts[col + '_rate'] = np.where(
            dir_counts['n_switchers_evaluable'] > 0,
            dir_counts[col] / dir_counts['n_switchers_evaluable'],
            np.nan
        )

    print(dir_counts.to_string(index=False))

    # Also print grand totals across low+high
    total_away  = switchers['toward_ai'].eq(False).sum()
    total_tow   = switchers['toward_ai'].eq(True).sum()
    total_eval  = len(switchers)
    print("\nTotals across low+high switchers (evaluable):")
    print(f"  away_from_AI: {total_away}")
    print(f"  toward_AI:    {total_tow}")
    print(f"  total:        {total_eval}")


In [ ]:
# stance_with_reliance_diagnostics.py
import pandas as pd
import numpy as np
from scipy import stats

STANCE_PATH = "comprehensive_stance_file.csv"   # your stance file (built earlier)
LOW_PATH    = "LIWC-22 Results - H-AI_Subjectivity_Study_Low_In___ - LIWC Analysis.csv"
HIGH_PATH   = "LIWC-22 Results - H-AI_Subjectivity_Study_High_I___ - LIWC Analysis (1).csv"
OUT_PATH    = "stance_with_reliance.csv"

# Text->numeric (Likert 1–5)
RELIANCE_MAP = {
    "Not at all": 1,
    "Too little": 2,
    "The right amount": 3,
    "Too much": 4,
    "Completely": 5
}

# ---- scenario -> (Q for perceived_reliance_1, Q for perceived_reliance_2)
mapping_low = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q88_1","Q49.1"),
    "sexism-3": ("Q89_1","Q49.2"),
    "sexism-4": ("Q90_1","Q49.3"),
    "aita-1":   ("Q91_1","Q49.4"),
    "aita-2":   ("Q92_1","Q49.5"),
    "aita-3":   ("Q93_1","Q49.6"),
    "aita-4":   ("Q94_1","Q49.8"),
}

mapping_high = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q69_1","Q70"),
    "sexism-3": ("Q71_1","Q72"),
    "sexism-4": ("Q73_1","Q74"),
    "aita-1":   ("Q75_1","Q76"),
    "aita-2":   ("Q77_1","Q78"),
    "aita-3":   ("Q79_1","Q80"),
    "aita-4":   ("Q81_1","Q82"),
}

def norm(s): 
    return str(s).strip()

def check_columns_exist(df, mapping, label):
    """Print whether each mapped column exists and its non-null count."""
    print(f"\n=== {label}: Column existence & coverage ===")
    for scen, (c1, c2) in mapping.items():
        c1_exists = c1 in df.columns
        c2_exists = c2 in df.columns
        n1 = df[c1].notna().sum() if c1_exists else 0
        n2 = df[c2].notna().sum() if c2_exists else 0
        print(f"{scen:9s} | {c1:<6} {'OK' if c1_exists else 'MISSING':>7} (n={n1:3d}) | "
              f"{c2:<6} {'OK' if c2_exists else 'MISSING':>7} (n={n2:3d})")

def build_reliance_long(df_raw, condition, mapping):
    """
    Return one row per (condition, response_id, scenario), keeping BOTH mapped measures.
    We’ll later derive:
      - reliance_num5  (Likert 1–5 from text)
      - reliance_0100  (0–100 slider from numeric)
    """
    id_col = "ResponseId" if "ResponseId" in df_raw.columns else "responseid"
    out = []
    for scen, (c1, c2) in mapping.items():
        # create missing columns so concat doesn’t fail
        if c1 not in df_raw.columns: df_raw[c1] = np.nan
        if c2 not in df_raw.columns: df_raw[c2] = np.nan
        block = pd.DataFrame({
            "condition": condition,
            "response_id": df_raw[id_col].astype(str).map(norm),
            "scenario": scen,
            "reliance_1_q": c1,
            "reliance_2_q": c2,
            "reliance_1_raw": df_raw[c1],
            "reliance_2_raw": df_raw[c2],
        })
        out.append(block)
    long_df = pd.concat(out, ignore_index=True)

    # Build BOTH numeric scales robustly:
    # 1) Likert 1–5 from text using RELIANCE_MAP on EITHER column (prefer 2, then 1)
    r1_text = long_df["reliance_1_raw"].astype(str).str.strip()
    r2_text = long_df["reliance_2_raw"].astype(str).str.strip()
    r1_num5 = r1_text.map(RELIANCE_MAP)
    r2_num5 = r2_text.map(RELIANCE_MAP)
    long_df["reliance_num5"] = r2_num5.combine_first(r1_num5)

    # 2) Slider 0–100 from numeric coercion on EITHER column (prefer 2, then 1)
    r1_0100 = pd.to_numeric(long_df["reliance_1_raw"], errors="coerce")
    r2_0100 = pd.to_numeric(long_df["reliance_2_raw"], errors="coerce")
    long_df["reliance_0100"] = r2_0100.combine_first(r1_0100)

    # Keep first per key
    agg = (long_df
           .groupby(["condition","response_id","scenario"], as_index=False)
           .agg({
               "reliance_num5": "first",
               "reliance_0100": "first",
               "reliance_1_q": "first",
               "reliance_2_q": "first",
               "reliance_1_raw": "first",
               "reliance_2_raw": "first",
           }))
    return agg

def main():
    # 1) Load stance file
    stance = pd.read_csv(STANCE_PATH, low_memory=False)

    # Normalize/ensure columns
    if "scenario_label" in stance.columns and "scenario" not in stance.columns:
        stance = stance.rename(columns={"scenario_label": "scenario"})
    stance = stance.rename(columns={
        "flip_first_final": "stance_switch",   # if needed
        "stance_ai_flip": "toward_ai"          # True if final==AI when flipped
    })
    stance["condition"]   = stance["condition"].astype(str).str.strip().str.lower()
    stance["response_id"] = stance["response_id"].astype(str).map(norm)
    stance["scenario"]    = stance["scenario"].astype(str).str.strip().str.lower()

    # Only low/high are relevant for AI reliance
    stance = stance[stance["condition"].isin(["low","high"])].copy()

    # 2) Load raw low/high & diagnostics
    low_raw  = pd.read_csv(LOW_PATH,  low_memory=False)
    high_raw = pd.read_csv(HIGH_PATH, low_memory=False)
    check_columns_exist(low_raw,  mapping_low,  "LOW")
    check_columns_exist(high_raw, mapping_high, "HIGH")

    # 3) Build reliance tables (both 1–5 & 0–100)
    rel_low  = build_reliance_long(low_raw,  "low",  mapping_low)
    rel_high = build_reliance_long(high_raw, "high", mapping_high)
    rel = pd.concat([rel_low, rel_high], ignore_index=True)

    # 4) Merge stance + reliance
    merged = stance.merge(
        rel,
        on=["condition","response_id","scenario"],
        how="left",
        validate="m:1"
    )

    # Coerce stance flags safely
    def to_bool(series):
        TRUE_SET  = {"true","t","1","yes","y","toward","towards"}
        FALSE_SET = {"false","f","0","no","n","away"}
        out = pd.Series(index=series.index, dtype="float")
        # bools
        is_bool = series.apply(lambda x: isinstance(x, (bool, np.bool_)))
        out.loc[is_bool] = series[is_bool].astype(bool)
        # numeric 0/1
        as_num = pd.to_numeric(series, errors="coerce")
        mask_num = as_num.isin([0, 1]) & ~series.isna()
        out.loc[mask_num] = as_num[mask_num].astype(bool)
        # strings
        s = series.astype(str).str.strip().str.lower()
        out.loc[s.isin(TRUE_SET)]  = True
        out.loc[s.isin(FALSE_SET)] = False
        out.loc[series.isna() | series.astype(str).str.strip().eq("")] = np.nan
        return out.astype("object")

    merged["stance_switch_bool"] = to_bool(merged.get("stance_switch"))
    merged["toward_ai_bool"]     = to_bool(merged.get("toward_ai"))

    # Save merged for reference
    merged.to_csv(OUT_PATH, index=False)
    print(f"\n✅ Saved: {OUT_PATH}")
    print("Rows:", len(merged), "| Unique response_ids:", merged["response_id"].nunique())

    # ---------------- Stats & Counts ----------------

    # 5) Counts per condition: total flips and flips toward AI
    print("\n=== Flip counts by condition ===")
    flip_counts = (merged.groupby("condition")["stance_switch_bool"]
                          .apply(lambda s: (s == True).sum())
                          .rename("n_flips"))
    print(flip_counts.to_string())

    print("\n=== Flips TOWARD AI (among switchers) by condition ===")
    # among rows where stance_switch_bool == True and toward_ai_bool is not NaN
    sw = merged[merged["stance_switch_bool"] == True].copy()
    toward_counts = (sw.groupby("condition")["toward_ai_bool"]
                       .apply(lambda s: (s == True).sum())
                       .rename("n_toward_AI"))
    print(toward_counts.to_string())

    # Also show totals of evaluable switchers (non-NaN toward/away)
    eval_counts = (sw.groupby("condition")["toward_ai_bool"]
                     .apply(lambda s: s.notna().sum())
                     .rename("n_switchers_evaluable"))
    print("\nEvaluable switchers by condition (have toward/away value):")
    print(eval_counts.to_string())

    # 6) Correlations: switch_toward_ai (0/1) vs BOTH reliance scales, per condition
    # Binary DV: switched *and* toward AI (1), otherwise 0
    merged["switch_toward_ai"] = ((merged["stance_switch_bool"] == True) & (merged["toward_ai_bool"] == True)).astype(int)

    # Ensure numeric 0–100 is float
    merged["reliance_0100"] = pd.to_numeric(merged["reliance_0100"], errors="coerce")
    # Likert 1–5 already numeric in reliance_num5

    def corr_block(df, cond, x_col="switch_toward_ai"):
        sub = df[df["condition"] == cond].copy()
        pairs = {
            "AI-toward (0/1) vs reliance (1–5 Likert)": (x_col, "reliance_num5"),
            "AI-toward (0/1) vs reliance (0–100 slider)": (x_col, "reliance_0100"),
        }
        print(f"\n=== Correlations in {cond} ===")
        any_printed = False
        for label, (x, y) in pairs.items():
            dat = sub[[x, y]].dropna()
            n = len(dat)
            if n < 3:
                print(f"{label}: not enough data (n={n})")
                continue
            r_p, p_p = stats.pearsonr(dat[x], dat[y])
            r_s, p_s = stats.spearmanr(dat[x], dat[y])
            print(f"{label} (n={n}): Pearson r={r_p:.3f}, p={p_p:.5f} | Spearman ρ={p_s:.3f}, p={p_s:.5f}")
            any_printed = True
        if not any_printed:
            print("No evaluable pairs.")

    corr_block(merged, "low")
    corr_block(merged, "high")


if __name__ == "__main__":
    main()


## Detail

In [ ]:
# detail_reliance_analyses_autodetect.py
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# -----------------------------
# Paths (edit if needed)
# -----------------------------
DETAIL_CSV  = "comprehensive_detail_lengths.csv"  # your detail-lengths file
LOW_PATH    = "LIWC-22 Results - H-AI_Subjectivity_Study_Low_In___ - LIWC Analysis.csv"
HIGH_PATH   = "LIWC-22 Results - H-AI_Subjectivity_Study_High_I___ - LIWC Analysis (1).csv"

# -----------------------------
# Reliance mappings (scenario -> (colA, colB))
# Order of (colA, colB) no longer matters; we auto-detect which is percent vs text per row.
# -----------------------------
mapping_low = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q88_1","Q49.1"),
    "sexism-3": ("Q89_1","Q49.2"),
    "sexism-4": ("Q90_1","Q49.3"),
    "aita-1":   ("Q91_1","Q49.4"),
    "aita-2":   ("Q92_1","Q49.5"),
    "aita-3":   ("Q93_1","Q49.6"),
    "aita-4":   ("Q94_1","Q49.8")
}

mapping_high = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q69_1","Q70"),
    "sexism-3": ("Q71_1","Q72"),
    "sexism-4": ("Q73_1","Q74"),
    "aita-1":   ("Q75_1","Q76"),
    "aita-2":   ("Q77_1","Q78"),
    "aita-3":   ("Q79_1","Q80"),
    "aita-4":   ("Q81_1","Q82")
}

RELIANCE_TEXT2NUM = {
    "Not at all": 1,
    "Too little": 2,
    "The right amount": 3,
    "Too much": 4,
    "Completely": 5
}

# -----------------------------
# Helpers
# -----------------------------
def _norm_id(x: str) -> str:
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s

def _norm_s(s: str) -> str:
    return str(s).strip().lower()

def _coerce_percent(val):
    try:
        x = float(val)
        return x if 0 <= x <= 100 else np.nan
    except Exception:
        return np.nan

def _coerce_text_choice(val):
    s = str(val).strip()
    return s if s in RELIANCE_TEXT2NUM else np.nan

def build_reliance_table_autodetect(path, cond, mapping):
    """
    For each scenario in `mapping`, two columns are provided.
    We autodetect per row which column is the % (0–100) and which is the 1–5 text label.
    Returns a tidy DF with: condition, response_id, scenario, reliance_scale (0–100),
    reliance_measure (text), reliance_measure_num (1–5).
    """
    raw = pd.read_csv(path, low_memory=False)
    id_col = "ResponseId" if "ResponseId" in raw.columns else ("response_id" if "response_id" in raw.columns else None)
    if id_col is None:
        raise ValueError(f"{path}: cannot find ResponseId/response_id")

    raw[id_col] = raw[id_col].astype(str).map(_norm_id)
    raw = raw.replace({"": np.nan, " ": np.nan})

    rows = []
    for scen, (c1, c2) in mapping.items():
        if (c1 not in raw.columns) and (c2 not in raw.columns):
            print(f"⚠️ {cond.upper()} survey missing both columns for {scen}: {c1} / {c2}")
            continue

        v1 = raw[c1] if c1 in raw.columns else pd.Series([np.nan]*len(raw))
        v2 = raw[c2] if c2 in raw.columns else pd.Series([np.nan]*len(raw))

        v1_pct  = v1.apply(_coerce_percent)
        v2_pct  = v2.apply(_coerce_percent)
        v1_text = v1.apply(_coerce_text_choice)
        v2_text = v2.apply(_coerce_text_choice)

        percent = np.where(~v1_pct.isna(), v1_pct, v2_pct)
        text    = np.where(pd.notna(v1_text), v1_text, v2_text)

        mask_any = (~pd.isna(percent)) | (pd.notna(text))
        if not mask_any.any():
            continue

        tmp = pd.DataFrame({
            "condition": cond,
            "response_id": raw.loc[mask_any, id_col].astype(str).map(_norm_id),
            "scenario": _norm_s(scen),
            "reliance_scale": percent[mask_any],
            "reliance_measure": pd.Series(text, index=raw.index)[mask_any]
        })
        rows.append(tmp)

    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["condition","response_id","scenario","reliance_scale","reliance_measure"]
    )

    tidy = (out.sort_values(["condition","response_id","scenario"])
              .groupby(["condition","response_id","scenario"], as_index=False)
              .agg({"reliance_scale":"first", "reliance_measure":"first"}))

    tidy["reliance_measure_num"] = tidy["reliance_measure"].map(RELIANCE_TEXT2NUM)

    print(f"{cond.upper()} reliance tidy shape:", tidy.shape)
    nn = tidy[["reliance_scale","reliance_measure","reliance_measure_num"]].notna().sum().to_dict()
    print(f"{cond.upper()} non-null counts:", nn)
    return tidy

def anova_and_posthoc(data, y, group="condition", alpha=0.05, label=""):
    sub = data[[y, group]].dropna()
    print(f"\n=== ANOVA: {label or y} by {group} ===")
    if sub[group].nunique() < 2:
        print("Not enough groups for ANOVA.")
        return
    model = ols(f"{y} ~ C({group})", data=sub).fit()
    aov = anova_lm(model, typ=2)
    print(aov)

def welch_low_vs_high(data, y, label=""):
    sub = data[data["condition"].isin(["low","high"])][[y,"condition"]].dropna()
    a = sub.loc[sub["condition"]=="low",  y].values
    b = sub.loc[sub["condition"]=="high", y].values
    t, p = stats.ttest_ind(a, b, equal_var=False)
    m_a, m_b = np.mean(a), np.mean(b)
    sd_a, sd_b = np.std(a, ddof=1), np.std(b, ddof=1)
    print(f"\n=== Welch t-test (low vs high): {label or y} ===")
    print(f"n_low={len(a)}, n_high={len(b)}")
    print(f"means: low={m_a:.2f}, high={m_b:.2f}")
    print(f"sds:   low={sd_a:.2f}, high={sd_b:.2f}")
    print(f"t={t:.3f}, p={p:.5f}")

def corr_block(data, cond, y_col, label):
    sub = data[data["condition"] == cond][[y_col, "reliance_scale", "reliance_measure_num"]]
    for meas, lab in [("reliance_scale","perceived % (0–100)"),
                      ("reliance_measure_num","perceived 1–5")]:
        d = sub[[y_col, meas]].dropna()
        n = len(d)
        if n < 3:
            print(f"{cond} | {label} vs {lab}: not enough data (n={n})")
            continue
        r_p, p_p = stats.pearsonr(d[y_col], d[meas])
        r_s, p_s = stats.spearmanr(d[y_col], d[meas])
        print(f"{cond} | {label} vs {lab}: "
              f"Pearson r={r_p:.3f}, p={p_p:.5f} | Spearman ρ={r_s:.3f}, p={p_s:.5f} (n={n})")

# -----------------------------
# Main
# -----------------------------
def main():
    # Load detail file
    df = pd.read_csv(DETAIL_CSV, low_memory=False)
    # Standardize keys
    df["condition"]   = df["condition"].astype(str).str.strip().str.lower()
    df["response_id"] = df["response_id"].astype(str).map(_norm_id)
    df["scenario"]    = df["scenario"].astype(str).map(_norm_s)

    # Quick sanity
    print("=== DETAIL sanity ===")
    print("rows:", len(df))
    print("conditions:", df["condition"].value_counts().to_dict())
    print("sample scenarios:", sorted(df["scenario"].dropna().unique())[:8])

    # Build reliance tables (auto-detect percent vs text)
    rel_low  = build_reliance_table_autodetect(LOW_PATH,  "low",  mapping_low)
    rel_high = build_reliance_table_autodetect(HIGH_PATH, "high", mapping_high)
    rel = pd.concat([rel_low, rel_high], ignore_index=True)

    # Merge with detail
    probe = df.merge(rel, on=["condition","response_id","scenario"], how="left", indicator=True)
    print("\n[MERGE] detail → reliance:", probe["_merge"].value_counts(dropna=False).to_dict())

    miss = probe.loc[probe["_merge"]=="left_only", ["condition","response_id","scenario"]].drop_duplicates()
    if not miss.empty:
        print("Examples of detail rows NOT found in reliance (first 20):")
        print(miss.head(20).to_string(index=False))

    merged = probe.drop(columns=["_merge"])

    # ----------------- Stats -----------------
    # ANOVAs on first–final deltas
    anova_and_posthoc(merged, "delta_chars_first_final", label="Δ chars (first - final)")
    anova_and_posthoc(merged, "delta_words_first_final", label="Δ words (first - final)")

    # Welch low vs high on final–AI deltas
    welch_low_vs_high(merged, "delta_chars_final_ai", label="Δ chars (final - AI)")
    welch_low_vs_high(merged, "delta_words_final_ai", label="Δ words (final - AI)")

    # Correlations with n
    print("\n=== Correlations: Final–AI deltas vs perceived reliance ===")
    for cond in ["low","high"]:
        corr_block(merged, cond, "delta_chars_final_ai", "Δ chars (final–AI)")
        corr_block(merged, cond, "delta_words_final_ai", "Δ words (final–AI)")

    # Save merged for reference
    merged.to_csv("detail_with_reliance_merged.csv", index=False)
    print("\n✅ Saved: detail_with_reliance_merged.csv")

if __name__ == "__main__":
    main()


## Language

In [ ]:

# stats_comprehensive_language.py
import pandas as pd
import numpy as np
from itertools import combinations
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

# ---------- Paths ----------
LANG_PATH = "comprehensive_language_file.csv"
LOW_PATH  = "LIWC-22 Results - H-AI_Subjectivity_Study_Low_In___ - LIWC Analysis.csv"
HIGH_PATH = "LIWC-22 Results - H-AI_Subjectivity_Study_High_I___ - LIWC Analysis (1).csv"
OUT_MERGED = "language_with_reliance.csv"

# ---------- Helpers ----------
def cohen_d_independent(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    sa2, sb2 = np.var(a, ddof=1), np.var(b, ddof=1)
    s_pooled = np.sqrt(((na-1)*sa2 + (nb-1)*sb2) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / s_pooled

def pairwise_welch_bh(df, value_col, group_col, alpha=0.05):
    groups = df[group_col].dropna().unique()
    pairs = list(combinations(groups, 2))
    results, pvals = [], []

    for g1, g2 in pairs:
        a = df.loc[df[group_col] == g1, value_col].dropna().values
        b = df.loc[df[group_col] == g2, value_col].dropna().values
        if len(a) < 2 or len(b) < 2:
            continue
        t, p = stats.ttest_ind(a, b, equal_var=False)
        d = cohen_d_independent(a, b)
        results.append({
            "pair": f"{g1} vs {g2}",
            "t": float(t), "p_raw": float(p),
            "cohens_d": float(d),
            "n1": int(len(a)), "n2": int(len(b))
        })
        pvals.append(p)

    if not pvals:
        return pd.DataFrame(results)

    rej, p_bh, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")
    for r, p_adj, reject in zip(results, p_bh, rej):
        r["p_bh"] = float(p_adj)
        r["reject_H0"] = bool(reject)

    return pd.DataFrame(results).sort_values("p_raw")

def norm(s):  # shared normalizer
    return str(s).strip().lower()

# ---------- PART A: Language stats ----------
df = pd.read_csv(LANG_PATH, low_memory=False)

# One row per (response_id, scenario)
df_clean = (
    df.sort_values(["response_id", "scenario"])
      .drop_duplicates(subset=["response_id", "scenario"], keep="first")
      .reset_index(drop=True)
)

print(f"Rows after clean: {len(df_clean)}")
print(f"Unique response_ids: {df_clean['response_id'].nunique()}")

# 1) First–Final cosine
print("\n=== Language: First–Final Cosine Similarity ===")
df_ff = df_clean[["condition","cosine_first_final_liwc"]].dropna()

if df_ff["condition"].nunique() >= 2 and df_ff.shape[0] >= 3:
    anova_model = ols("cosine_first_final_liwc ~ C(condition)", data=df_ff).fit()
    anova_table = sm.stats.anova_lm(anova_model, typ=2)
    print("ANOVA:\n", anova_table, "\n")
else:
    print("ANOVA skipped (not enough data).")

means = df_ff.groupby("condition")["cosine_first_final_liwc"].agg(["mean","std","count"])
print("Group means:\n", means, "\n")

pw_ff = pairwise_welch_bh(df_ff, "cosine_first_final_liwc", "condition")
if not pw_ff.empty:
    print("Pairwise Welch tests (BH corrected):")
    print(pw_ff.to_string(index=False))
else:
    print("Pairwise tests skipped (not enough group pairs).")

# 2) Final–AI cosine (low vs high)
print("\n=== Language: Final–AI Cosine Similarity (low vs high) ===")
df_ai = df_clean[df_clean["condition"].isin(["low","high"])].dropna(subset=["cosine_final_ai_liwc"]).copy()
print("Group sizes:\n", df_ai.groupby("condition")["cosine_final_ai_liwc"].count(), "\n")
desc = df_ai.groupby("condition")["cosine_final_ai_liwc"].agg(["mean","std","count"])
print("Descriptives:\n", desc, "\n")

if all(c in df_ai["condition"].unique() for c in ["low","high"]):
    low_vals  = df_ai.loc[df_ai["condition"]=="low","cosine_final_ai_liwc"].to_numpy(float)
    high_vals = df_ai.loc[df_ai["condition"]=="high","cosine_final_ai_liwc"].to_numpy(float)
    if len(low_vals) >= 2 and len(high_vals) >= 2:
        t, p = stats.ttest_ind(low_vals, high_vals, equal_var=False)
        d = cohen_d_independent(low_vals, high_vals)
        print(f"Welch's t-test: t={t:.3f}, p={p:.5f}, d={d:.3f}")
    else:
        print("Welch t-test skipped (not enough rows per group).")
else:
    print("Welch t-test skipped (missing one of the groups).")

# 3) Within-condition comparison
print("\n=== Language: Within-condition comparisons (first–final vs final–AI) ===")
df_both = (
    df_clean[df_clean["condition"].isin(["low","high"])]
      .dropna(subset=["cosine_first_final_liwc","cosine_final_ai_liwc"])
      .copy()
)
df_both["delta"] = df_both["cosine_first_final_liwc"] - df_both["cosine_final_ai_liwc"]

for cond in ["low","high"]:
    sub = df_both[df_both["condition"]==cond]
    x, y = sub["cosine_first_final_liwc"].to_numpy(float), sub["cosine_final_ai_liwc"].to_numpy(float)
    if len(sub) >= 2:
        t, p = stats.ttest_rel(x, y, nan_policy="omit")
        diff = x - y
        dz = np.mean(diff) / np.std(diff, ddof=1)
        print(f"[{cond}] Paired t-test (n={len(sub)}): t={t:.3f}, p={p:.5f}, dz={dz:.3f}")
    else:
        print(f"[{cond}] Paired test skipped (n={len(sub)}).")

# Δ between conditions
low_d  = df_both.loc[df_both["condition"]=="low","delta"].to_numpy(float)
high_d = df_both.loc[df_both["condition"]=="high","delta"].to_numpy(float)
if len(low_d) >= 2 and len(high_d) >= 2:
    t, p = stats.ttest_ind(low_d, high_d, equal_var=False)
    d = cohen_d_independent(low_d, high_d)
    print("\nΔ between conditions:")
    print(f"  mean Δ (low)={np.mean(low_d):.3f}, mean Δ (high)={np.mean(high_d):.3f}")
    print(f"  Welch's t={t:.3f}, p={p:.5f}, d={d:.3f}")
else:
    print("\nΔ between conditions: skipped (not enough data).")

# ---------- PART B: Reliance merge (EXACTLY as your working pattern, with DIAGNOSTICS) ----------
RELIANCE_MAP = {
    "Not at all": 1,
    "Too little": 2,
    "The right amount": 3,
    "Too much": 4,
    "Completely": 5
}

mapping_low = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q88_1","Q49.1"),
    "sexism-3": ("Q89_1","Q49.2"),
    "sexism-4": ("Q90_1","Q49.3"),
    "aita-1":   ("Q91_1","Q49.4"),
    "aita-2":   ("Q92_1","Q49.5"),
    "aita-3":   ("Q93_1","Q49.6"),
    "aita-4":   ("Q94_1","Q49.8"),
}
mapping_high = {
    "sexism-1": ("Q47_1","Q49"),
    "sexism-2": ("Q69_1","Q70"),
    "sexism-3": ("Q71_1","Q72"),
    "sexism-4": ("Q73_1","Q74"),
    "aita-1":   ("Q75_1","Q76"),
    "aita-2":   ("Q77_1","Q78"),
    "aita-3":   ("Q79_1","Q80"),
    "aita-4":   ("Q81_1","Q82"),
}

def build_reliance_table(path, cond, mapping):
    df_raw = pd.read_csv(path, low_memory=False)
    id_col = "ResponseId" if "ResponseId" in df_raw.columns else "responseid"

    out = []
    for scen, (c1, c2) in mapping.items():
        for _, row in df_raw.iterrows():
            rid = str(row[id_col]).strip()
            for col in (c1, c2):
                if col not in df_raw.columns:
                    continue
                val = row[col]
                if pd.isna(val):
                    continue
                out.append({
                    "condition": cond,
                    "response_id": rid,
                    "scenario": scen,
                    "column_id": col,
                    "reliance_measure": val,                     # raw (text or number as string)
                    "reliance_scale": pd.to_numeric(val, errors="coerce"),  # slider if numeric
                    "reliance_measure_num": RELIANCE_MAP.get(str(val).strip(), np.nan)  # Likert 1–5 if text
                })
    return pd.DataFrame(out)

# Build long table & dedupe exactly like your working code
rel_low  = build_reliance_table(LOW_PATH,  "low",  mapping_low)
rel_high = build_reliance_table(HIGH_PATH, "high", mapping_high)
rel = pd.concat([rel_low, rel_high], ignore_index=True)
print("\n[REL] long shape (before dedupe):", rel.shape)

rel = rel.drop_duplicates(subset=["condition","response_id","scenario","column_id"])
print("[REL] long shape (after dedupe):", rel.shape)

# Aggregate to one row per (cond, id, scenario) *before* merge
rel_agg = (rel
    .groupby(["condition","response_id","scenario"], as_index=False)
    .agg({
        "reliance_measure_num": lambda s: pd.to_numeric(s, errors="coerce").dropna().max() if pd.to_numeric(s, errors="coerce").notna().any() else np.nan,
        "reliance_scale":       lambda s: pd.to_numeric(s, errors="coerce").dropna().max() if pd.to_numeric(s, errors="coerce").notna().any() else np.nan
    })
)

# ---------- DIAGNOSTICS: keys & overlap ----------
comp = df_clean.copy()
for c in ["condition","response_id","scenario"]:
    comp[c] = comp[c].astype(str).str.strip().str.lower()
for c in ["condition","response_id","scenario"]:
    rel_agg[c] = rel_agg[c].astype(str).str.strip().str.lower()

print("\n[DIAG] comp shape:", comp.shape)
print("[DIAG] rel_agg shape:", rel_agg.shape)

overlap_ids = len(set(comp["response_id"]) & set(rel_agg["response_id"]))
overlap_cond_id = len(set(zip(comp["condition"],comp["response_id"])) & set(zip(rel_agg["condition"],rel_agg["response_id"])))
print(f"[DIAG] overlap response_id: {overlap_ids} / {comp['response_id'].nunique()}")
print(f"[DIAG] overlap (condition,response_id): {overlap_cond_id} / {comp[['condition','response_id']].drop_duplicates().shape[0]}")

print("\n[DIAG] scenario samples in comp:", sorted(comp["scenario"].unique())[:8])
print("[DIAG] scenario samples in rel_agg:", sorted(rel_agg["scenario"].unique()))

# Merge with indicator
merged = comp.merge(
    rel_agg,
    on=["condition","response_id","scenario"],
    how="left",
    indicator=True
)
print("\n[MERGE] indicator:", merged["_merge"].value_counts().to_dict())

if (merged["_merge"] == "left_only").any():
    print("\n[MERGE] examples of left_only (first 12):")
    print(merged.loc[merged["_merge"]=="left_only", ["condition","response_id","scenario"]]
                .drop_duplicates()
                .head(12)
                .to_string(index=False))

# Save merged file
merged.drop(columns=["_merge"], inplace=True)
merged.to_csv(OUT_MERGED, index=False)
print(f"\n✅ Saved: {OUT_MERGED}")

# ---------- Post-merge coverage ----------
cov = (merged.assign(any_reliance = merged[["reliance_measure_num","reliance_scale"]].notna().any(axis=1))
              .groupby("condition")["any_reliance"].agg(["sum","count"]))
print("\n[POST-MERGE] rows with any reliance value (per condition):")
print(cov)

# ---------- PART C: Correlations vs BOTH reliance measures ----------
def corr_report_both_measures(data, condition):
    sub = data[(data["condition"] == condition)].copy()

    pairs = [
        ("cosine_final_ai_liwc", "reliance_measure_num", "AI-final vs reliance (1–5 Likert)"),
        ("cosine_final_ai_liwc", "reliance_scale",       "AI-final vs reliance (0–100 slider)"),
    ]

    print(f"\n=== Language correlations in {condition} ===")
    for xcol, ycol, label in pairs:
        d = sub[[xcol, ycol]].dropna()
        n = len(d)
        print(f"{label}: n={n}")
        if n < 3:
            print("  Not enough data.")
            continue
        x = d[xcol].to_numpy(float)
        y = d[ycol].to_numpy(float)
        r_p, p_p = stats.pearsonr(x, y)
        r_s, p_s = stats.spearmanr(x, y)
        print(f"  Pearson r = {r_p:.3f}, p = {p_p:.5f}")
        print(f"  Spearman ρ = {r_s:.3f}, p = {p_s:.5f}")

corr_report_both_measures(merged, "low")
corr_report_both_measures(merged, "high")


# Updated statistical analysis (addresses reviewer comments)

This section updates the statistical analysis to address reviewer feedback **without changing the dataset-fixing / file-building steps above**.

## What changed and why

1. **Use mixed-effects models (one model per metric)**  
   - The data are clustered: each participant (`response_id`) contributes multiple scenarios.  
   - We replace ANOVA + many follow-up t-tests with a single mixed model per metric that includes:
     - random intercept for `response_id`
     - random intercept for `scenario`
   - This directly addresses the “design mismatch / non-independence” concern.

2. **Reduce the number of t-tests by modeling FF vs FA jointly**  
   - For each metric, we reshape the data to a *long* format with `pair_type ∈ {FF, FA}`:
     - FF = first–final
     - FA = final–AI
   - We fit: `value ~ condition * pair_type + (1|response_id) + (1|scenario)`
   - This single model yields:
     - whether FF differs from FA within each condition (previously a paired t-test)
     - whether FA differs across conditions (previously another t-test)
     - whether the FF–FA gap differs by condition (interaction)

3. **Effect sizes + confidence intervals**
   - For each planned contrast, we report:
     - point estimate
     - **cluster bootstrap 95% CI** (resampling `response_id`)
     - standardized effect size computed from participant-level aggregated values (Cohen’s d) as a readable summary

4. **Multiple comparisons (Holm correction)**
   - We define a **primary family** of tests across modalities (Theme, LIWC/Language, Detail, Stance).
   - Within each family, we apply **Holm correction** and report adjusted p-values.

5. **Stance “Depends” issue**
   - We recommend (and implement in the stance section below) a *distance-to-AI* approach using an ordinal mapping:
     - No = 0, Depends = 0.5, Yes = 1
   - This allows partial movement toward AI (e.g., No→Depends when AI=Yes) instead of forcing “toward AI = 0”.


In [ ]:
# Imports used in the updated analysis section
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from itertools import product
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the comprehensive files created by the "fixing datasets" pipeline above.
# If any are missing, re-run the earlier build cells.

THEME_PATH   = "comprehensive_theme_file.csv"
LANG_PATH    = "comprehensive_language_file.csv"
DETAIL_PATH  = "comprehensive_detail_lengths.csv"
STANCE_PATH  = "comprehensive_stance_file.csv"

def must_exist(p):
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing {p}. Re-run the earlier build cells (the file-fixing part).")

must_exist(THEME_PATH)
must_exist(LANG_PATH)
must_exist(DETAIL_PATH)
must_exist(STANCE_PATH)

theme_df  = pd.read_csv(THEME_PATH,  low_memory=False)
lang_df   = pd.read_csv(LANG_PATH,   low_memory=False)
detail_df = pd.read_csv(DETAIL_PATH, low_memory=False)
stance_df = pd.read_csv(STANCE_PATH, low_memory=False)

print("Loaded:", THEME_PATH, LANG_PATH, DETAIL_PATH, STANCE_PATH)


In [ ]:
# Helper: consistent de-duplication (one row per response_id × scenario)
def dedupe_response_scenario(df, response_col="response_id", scenario_col="scenario"):
    df = df.copy()
    df[response_col] = df[response_col].astype(str).str.strip()
    df[scenario_col] = df[scenario_col].astype(str).str.strip()
    df["condition"]  = df["condition"].astype(str).str.strip().str.lower()
    return (df.sort_values([response_col, scenario_col])
              .drop_duplicates(subset=[response_col, scenario_col], keep="first")
              .reset_index(drop=True))

theme_df  = dedupe_response_scenario(theme_df)
lang_df   = dedupe_response_scenario(lang_df)
detail_df = dedupe_response_scenario(detail_df)
stance_df = dedupe_response_scenario(stance_df)

print("Rows (theme/lang/detail/stance):", len(theme_df), len(lang_df), len(detail_df), len(stance_df))
print("Conditions:", theme_df["condition"].value_counts().to_dict())


In [ ]:
# --- Core idea: model FF vs FA jointly in one mixed model per metric ---
# We create long format with pair_type in {"FF","FA"} and fit:
#   value ~ C(condition) * C(pair_type) + (1|response_id) + (1|scenario)
#
# This replaces:
# - ANOVA on FF across conditions
# - t-test FA across conditions
# - paired t-test FF vs FA within condition
# - t-test on delta across conditions
#
# We use MixedLM with:
# - groups = response_id (participant random intercept)
# - scenario variance component (random intercept for scenario)

def make_long_pair(df, response_col="response_id", scenario_col="scenario",
                   ff_col="cosine_first_final", fa_col="cosine_final_ai",
                   value_name="value"):
    out = df[[ "condition", response_col, scenario_col, ff_col, fa_col ]].copy()
    out = out.dropna(subset=[ff_col, fa_col])
    long = out.melt(
        id_vars=["condition", response_col, scenario_col],
        value_vars=[ff_col, fa_col],
        var_name="pair_type",
        value_name=value_name
    )
    long["pair_type"] = long["pair_type"].map({ff_col:"FF", fa_col:"FA"})
    long["condition"] = long["condition"].astype("category")
    long["pair_type"] = long["pair_type"].astype("category")
    return long

def fit_mixed_pair(long_df, response_col="response_id", scenario_col="scenario", value_col="value"):
    vc = {"scenario": f"0 + C({scenario_col})"}
    model = smf.mixedlm(
        f"{value_col} ~ C(condition) * C(pair_type)",
        long_df,
        groups=long_df[response_col],
        vc_formula=vc,
        re_formula="1"
    )
    res = model.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
    return res

# Cluster bootstrap for a planned contrast function that maps fitted results -> scalar estimate
def cluster_bootstrap_ci(long_df, fit_fn, contrast_fn, cluster_col="response_id",
                         n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    clusters = long_df[cluster_col].unique()
    boot = []
    for _ in range(n_boot):
        samp = rng.choice(clusters, size=len(clusters), replace=True)
        bdf = pd.concat([long_df[long_df[cluster_col]==c] for c in samp], ignore_index=True)
        try:
            res = fit_fn(bdf)
            boot.append(float(contrast_fn(res)))
        except Exception:
            continue
    if len(boot) < max(50, n_boot*0.3):
        return (np.nan, np.nan, np.nan, len(boot))
    boot = np.array(boot)
    return (float(np.percentile(boot, 2.5)),
            float(np.percentile(boot, 97.5)),
            float(np.mean(boot)),
            len(boot))

# Simple Cohen's d on participant-aggregated values (readable effect size)
def cohen_d_from_participant_means(df, response_col, value_col, cond_a, cond_b):
    a = df[df["condition"]==cond_a].groupby(response_col)[value_col].mean()
    b = df[df["condition"]==cond_b].groupby(response_col)[value_col].mean()
    a, b = a.dropna().values, b.dropna().values
    if len(a)<2 or len(b)<2:
        return np.nan
    sa2, sb2 = np.var(a, ddof=1), np.var(b, ddof=1)
    sp = np.sqrt(((len(a)-1)*sa2 + (len(b)-1)*sb2) / (len(a)+len(b)-2))
    return float((np.mean(a) - np.mean(b)) / sp) if sp>0 else np.nan


In [ ]:
print(detail_df)

## Updated analyses per modality (Theme, Language/LIWC, Detail, Stance)

For each modality we:
1. Create a long dataset with FF vs FA (`pair_type`).
2. Fit one mixed model: `value ~ condition * pair_type + (1|response_id) + (1|scenario)`.
3. Extract planned contrasts with bootstrap 95% CIs.
4. Apply Holm correction across the primary family of contrasts.


In [ ]:
# Run the joint mixed model for Theme / Language / Detail cosine metrics

def run_joint_metric(df, ff_col, fa_col, label):
    long = make_long_pair(df, ff_col=ff_col, fa_col=fa_col)
    res = fit_mixed_pair(long)
    print(f"\n[{label}] Mixed model fit complete. N rows (long)={len(long)}")
    print(res.summary())

    # Planned contrasts:
    # 1) Within each condition: FA - FF (is final closer to AI than to initial?)
    # 2) Between conditions on FA (low vs high; baseline vs high; etc.) [only if present]
    # 3) Interaction: does (FA-FF) differ by condition? (condition × pair_type)

    # Build a prediction function for each contrast using model params
    # We'll use model's design matrix via patsy in statsmodels by predicting on small new dataframes.
    def pred(mean_condition, mean_pair):
        tmp = pd.DataFrame({
            "condition":[mean_condition],
            "pair_type":[mean_pair],
            "response_id":[long["response_id"].iloc[0]],
            "scenario":[long["scenario"].iloc[0]],
            "value":[0.0]
        })
        tmp["condition"] = tmp["condition"].astype(long["condition"].dtype)
        tmp["pair_type"] = tmp["pair_type"].astype(long["pair_type"].dtype)
        # statsmodels MixedLM doesn't have .predict with new vc easily; use fixed effects only:
        fe = res.fe_params
        # Build design row using patsy from model formula
        import patsy
        X = patsy.dmatrix("C(condition) * C(pair_type)", tmp, return_type="dataframe")
        return float(np.dot(X.iloc[0].values, fe.values))

    conditions = sorted(long["condition"].cat.categories.tolist())

    # Collect contrasts
    rows = []
    # within-condition FA-FF
    for c in conditions:
        est = pred(c, "FA") - pred(c, "FF")
        # bootstrap CI
        ci_low, ci_high, boot_mean, n_ok = cluster_bootstrap_ci(
            long, fit_mixed_pair,
            lambda r: ( # contrast fn using same fixed-effect pred approach
                (lambda _r=r: (
                    # quick fixed-effect predictions inside bootstrap
                    (lambda cond=c: (
                        float(np.dot(__import__("patsy").dmatrix("C(condition) * C(pair_type)",
                            pd.DataFrame({"condition":[cond],"pair_type":["FA"],"response_id":[long["response_id"].iloc[0]],"scenario":[long["scenario"].iloc[0]],"value":[0.0]}),
                            return_type="dataframe").iloc[0].values, _r.fe_params.values))
                        -
                        float(np.dot(__import__("patsy").dmatrix("C(condition) * C(pair_type)",
                            pd.DataFrame({"condition":[cond],"pair_type":["FF"],"response_id":[long["response_id"].iloc[0]],"scenario":[long["scenario"].iloc[0]],"value":[0.0]}),
                            return_type="dataframe").iloc[0].values, _r.fe_params.values))
                    ))()
                ))()
            ),
            n_boot=500, seed=123
        )
        rows.append({
            "metric": label,
            "contrast": f"FA-FF within {c}",
            "estimate": est,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "boot_n": n_ok,
            "effect_size_d": np.nan
        })

    # between-condition on FA (pairwise)
    if len(conditions) >= 2:
        for i in range(len(conditions)):
            for j in range(i+1, len(conditions)):
                c1, c2 = conditions[i], conditions[j]
                est = pred(c2, "FA") - pred(c1, "FA")
                # effect size using participant means of FA only
                fa_only = long[long["pair_type"]=="FA"].copy()
                d = cohen_d_from_participant_means(fa_only, "response_id", "value", c2, c1)
                ci_low, ci_high, boot_mean, n_ok = cluster_bootstrap_ci(
                    long, fit_mixed_pair,
                    lambda r: (
                        # pred(c2,FA)-pred(c1,FA) fixed effects
                        (lambda _r=r: (
                            float(np.dot(__import__("patsy").dmatrix("C(condition) * C(pair_type)",
                                pd.DataFrame({"condition":[c2],"pair_type":["FA"],"response_id":[long["response_id"].iloc[0]],"scenario":[long["scenario"].iloc[0]],"value":[0.0]}),
                                return_type="dataframe").iloc[0].values, _r.fe_params.values))
                            -
                            float(np.dot(__import__("patsy").dmatrix("C(condition) * C(pair_type)",
                                pd.DataFrame({"condition":[c1],"pair_type":["FA"],"response_id":[long["response_id"].iloc[0]],"scenario":[long["scenario"].iloc[0]],"value":[0.0]}),
                                return_type="dataframe").iloc[0].values, _r.fe_params.values))
                        ))()
                    ),
                    n_boot=500, seed=321
                )
                rows.append({
                    "metric": label,
                    "contrast": f"FA {c2}-{c1}",
                    "estimate": est,
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                    "boot_n": n_ok,
                    "effect_size_d": d
                })

    out = pd.DataFrame(rows)
    return out

theme_out = run_joint_metric(theme_df, "cosine_first_final", "cosine_final_ai", "Theme")
lang_out  = run_joint_metric(lang_df,  "cosine_first_final", "cosine_final_ai", "Language_LIWC")
detail_out= run_joint_metric(detail_df,"cosine_first_final", "cosine_final_ai", "Detail") if "cosine_first_final" in detail_df.columns else None

results = pd.concat([theme_out, lang_out] + ([detail_out] if detail_out is not None else []), ignore_index=True)

# Holm correction across contrasts within each metric family (separately)
results["p_raw_placeholder"] = np.nan  # bootstrap CI approach; we primarily report CI, but provide placeholder for future
results.to_csv(OUT_DIR / "updated_joint_mixed_results_theme_lang_detail.csv", index=False)
print("Saved:", OUT_DIR / "updated_joint_mixed_results_theme_lang_detail.csv")
display(results)


### Stance (addresses “Depends”)

We avoid coding “toward AI” as 0 for Depends by mapping stances to an ordinal scale:
- No = 0, Depends = 0.5, Yes = 1

Then:
- distance_first = |first - AI|
- distance_final = |final - AI|
- stance_reliance = distance_first - distance_final

We analyze stance using:
- **GEE logistic** for binary outcomes (changed; aligned; moved-toward-AI)
- **MixedLM** for the continuous distance-based stance_reliance as a robustness check


In [ ]:
# Stance updated analysis
# Expect stance_df to have: stance_first, stance_final, ai_stance_final_norm (or similar).
# We'll compute numeric stances and the distance-based reliance, then model jointly similar to other metrics.

def pick(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        lc = c.lower()
        if lc in lower_map:
            return lower_map[lc]
    raise KeyError(f"None of {candidates} found. Available: {list(df.columns)[:40]}")

cond_col = pick(stance_df, ["condition"])
resp_col = pick(stance_df, ["response_id"])
scen_col = pick(stance_df, ["scenario"])
first_col = pick(stance_df, ["stance_first"])
final_col = pick(stance_df, ["stance_final"])
ai_col = pick(stance_df, ["ai_stance_final_norm","ai_stance_final"])

# Normalize strings
def norm_stance(s):
    if pd.isna(s): return np.nan
    t = str(s).strip().lower()
    if t in {"yes","sexist"}: return "Yes"
    if t in {"no","not sexist","not_sexist","notsexist"}: return "No"
    if "depend" in t: return "Depends"
    return np.nan

S_MAP = {"No":0.0, "Depends":0.5, "Yes":1.0}

sd = stance_df[[cond_col, resp_col, scen_col, first_col, final_col, ai_col]].copy()
sd.rename(columns={cond_col:"condition", resp_col:"response_id", scen_col:"scenario"}, inplace=True)
sd["condition"] = sd["condition"].astype(str).str.strip().str.lower()
sd["response_id"] = sd["response_id"].astype(str).str.strip()
sd["scenario"] = sd["scenario"].astype(str).str.strip()

sd["first_norm"] = sd[first_col].apply(norm_stance)
sd["final_norm"] = sd[final_col].apply(norm_stance)
sd["ai_norm"] = sd[ai_col].apply(norm_stance)

sd["first_num"] = sd["first_norm"].map(S_MAP)
sd["final_num"] = sd["final_norm"].map(S_MAP)
sd["ai_num"]    = sd["ai_norm"].map(S_MAP)

sd = sd.dropna(subset=["first_num","final_num","ai_num"])
sd = dedupe_response_scenario(sd, response_col="response_id", scenario_col="scenario")

sd["dist_first_ai"] = (sd["first_num"] - sd["ai_num"]).abs()
sd["dist_final_ai"] = (sd["final_num"] - sd["ai_num"]).abs()
sd["stance_reliance"] = sd["dist_first_ai"] - sd["dist_final_ai"]  # positive = moved closer

# Binary moved-toward-AI
sd["stance_toward_ai"] = (sd["stance_reliance"] > 0).astype(int)
sd["stance_aligned"] = (sd["dist_final_ai"] == 0).astype(int)
sd["stance_changed"] = (sd["first_num"] != sd["final_num"]).astype(int)

print(sd[["condition","stance_reliance","stance_toward_ai","stance_aligned","stance_changed"]].groupby("condition").mean())

# MixedLM on stance_reliance (continuous)
long_stance = sd.copy()
long_stance["condition"] = long_stance["condition"].astype("category")
vc = {"scenario": "0 + C(scenario)"}
m = smf.mixedlm("stance_reliance ~ C(condition)", long_stance, groups=long_stance["response_id"], vc_formula=vc, re_formula="1")
res = m.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
print(res.summary())

# GEE on binary moved-toward-AI (more interpretable)
gee = smf.gee("stance_toward_ai ~ C(condition) + C(scenario)",
              groups="response_id", data=sd,
              family=sm.families.Binomial(),
              cov_struct=sm.cov_struct.Exchangeable()).fit()
print(gee.summary())

# Bootstrap CI for low vs high difference in stance_reliance (if both exist)
conds = sorted(sd["condition"].unique().tolist())
if "low" in conds and "high" in conds:
    # cluster bootstrap on response_id for difference in means (high-low)
    rng = np.random.default_rng(0)
    clusters = sd["response_id"].unique()
    boot = []
    for _ in range(1000):
        samp = rng.choice(clusters, size=len(clusters), replace=True)
        b = pd.concat([sd[sd.response_id==c] for c in samp], ignore_index=True)
        m_hi = b[b.condition=="high"]["stance_reliance"].mean()
        m_lo = b[b.condition=="low"]["stance_reliance"].mean()
        boot.append(m_hi - m_lo)
    ci = (np.percentile(boot,2.5), np.percentile(boot,97.5))
    print("Bootstrap 95% CI for (high-low) stance_reliance:", ci)


## Notes if something seems missing

- This notebook assumes the earlier cells still produce the four `comprehensive_*_file.csv` outputs.
- If the current version you uploaded has missing file-reading cells or path definitions, the only required fix is to ensure those `comprehensive_*` CSVs exist in the working directory before running this updated section.

## Where results are saved
- `outputs/updated_joint_mixed_results_theme_lang_detail.csv`
- (stance results are printed; you can export similarly if desired)


## Family-wise Holm corrections (Directional / Final alignment / Revision magnitude)

This section **adds Holm corrections by family**, using the updated mixed-effects approach.

### What “different numbers of conditions” means for correction
- **Directional reliance** and **Final alignment**: confirmatory contrast is **Low vs High only** → one p-value per modality.  
  Holm is applied across the **4 modalities** (stance, theme, language/LIWC, detail).
- **Revision magnitude**: includes **Baseline vs Low vs High** → we use an **omnibus condition test** per modality (Wald test of both Baseline and High vs Low coefficients).  
  Holm is applied across the **4 omnibus p-values** (one per modality).  
  If you run post-hoc pairwise contrasts, apply Holm **within each modality** across the 3 pairwise comparisons (reported as secondary/exploratory).

### Output
Creates and saves:
- `outputs/holm_family_directional.csv`
- `outputs/holm_family_final_alignment.csv`
- `outputs/holm_family_revision_omnibus.csv`


In [ ]:
# Holm-by-family tables (MixedLM + cluster bootstrap CIs)

import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- File inputs produced by the "fixing the files" section ----
THEME_PATH = Path("comprehensive_theme_file.csv")
LANG_PATH  = Path("comprehensive_language_file.csv")
DETAIL_PATH= Path("comprehensive_detail_file.csv")
STANCE_PATH= Path("comprehensive_stance_file.csv")

missing = [p for p in [THEME_PATH, LANG_PATH, DETAIL_PATH, STANCE_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing expected comprehensive files. The fixing/build section likely didn't run or filenames differ:\n"
        + "\n".join([str(p) for p in missing])
    )

def cluster_bootstrap_ci(df, func, cluster_col="response_id", n_boot=800, seed=123):
    rng = np.random.default_rng(seed)
    clusters = df[cluster_col].dropna().unique()
    if len(clusters) < 5:
        return (np.nan, np.nan)
    vals = []
    for _ in range(n_boot):
        samp = rng.choice(clusters, size=len(clusters), replace=True)
        boot = pd.concat([df[df[cluster_col]==cid] for cid in samp], ignore_index=True)
        vals.append(func(boot))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)

def fit_mixedlm(df, formula, group_col="response_id", vc_scenario=True):
    d = df.copy()
    d["condition"] = d["condition"].astype("category")
    d["scenario"] = d["scenario"].astype("category")
    if vc_scenario:
        vc = {"scenario": "0 + C(scenario)"}
        model = smf.mixedlm(formula, d, groups=d[group_col], vc_formula=vc, re_formula="1")
    else:
        model = smf.mixedlm(formula, d, groups=d[group_col], re_formula="1")
    res = model.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
    return res

def set_condition_order(df):
    # Low is the reference
    existing = set(df["condition"].astype(str).unique())
    ordered = ["Low"]
    for x in ["High","Baseline"]:
        if x in existing:
            ordered.append(x)
    df["condition"] = pd.Categorical(df["condition"], categories=ordered, ordered=True)
    return df

def wald_omnibus_condition(res, terms):
    idx = list(res.params.index)
    present = [t for t in terms if t in idx]
    if not present:
        return np.nan
    R = np.zeros((len(present), len(idx)))
    for i,t in enumerate(present):
        R[i, idx.index(t)] = 1.0
    return float(res.wald_test(R).pvalue)

def load_metric(path, metric_label, ff_col, fa_col):
    df = pd.read_csv(path)
    required = ["response_id","scenario","condition", ff_col, fa_col]
    miss = [c for c in required if c not in df.columns]
    if miss:
        raise ValueError(f"{metric_label}: missing columns {miss} in {path}")
    out = df[["response_id","scenario","condition", ff_col, fa_col]].copy()
    out["metric"] = metric_label
    out = out.rename(columns={ff_col:"FF", fa_col:"FA"})
    return out

# Theme/language: cosine similarities
theme = load_metric(THEME_PATH, "theme", ff_col="first_final_cosine", fa_col="final_ai_cosine")
lang  = load_metric(LANG_PATH,  "language", ff_col="first_final_cosine", fa_col="final_ai_cosine")

# Detail file can vary; if these columns differ, update ff_col/fa_col here.
detail = load_metric(DETAIL_PATH, "detail_words", ff_col="delta_words_first_final", fa_col="delta_words_final_ai")

# Stance: distance-based scores
stance = pd.read_csv(STANCE_PATH)
need = ["response_id","scenario","condition","first_stance","final_stance","ai_stance"]
miss = [c for c in need if c not in stance.columns]
if miss:
    raise ValueError(f"stance: missing {miss} in {STANCE_PATH}")

stance_map = {"No":0.0, "Depends":0.5, "Yes":1.0}
def map_s(x): return stance_map.get(str(x).strip(), np.nan)

stance["first_num"] = stance["first_stance"].map(map_s)
stance["final_num"] = stance["final_stance"].map(map_s)
stance["ai_num"]    = stance["ai_stance"].map(map_s)

stance["FF"] = (stance["final_num"] - stance["first_num"]).abs()
stance["FA"] = (stance["final_num"] - stance["ai_num"]).abs()
stance["R"]  = (stance["first_num"] - stance["ai_num"]).abs() - (stance["final_num"] - stance["ai_num"]).abs()

stance_long = stance[["response_id","scenario","condition","FF","FA","R"]].copy()
stance_long["metric"] = "stance"

def make_pair_long(df_metric):
    dfm = df_metric.copy()
    dfm = set_condition_order(dfm)
    long = pd.melt(
        dfm,
        id_vars=["response_id","scenario","condition","metric"],
        value_vars=["FF","FA"],
        var_name="pair_type",
        value_name="value"
    )
    long["pair_type"] = pd.Categorical(long["pair_type"], categories=["FF","FA"], ordered=True)
    return long

theme_long  = make_pair_long(theme)
lang_long   = make_pair_long(lang)
detail_long = make_pair_long(detail)

# ---------- Directional family (Low vs High only) ----------
def directional_test(long_df, metric_name):
    sub = long_df[(long_df["condition"].isin(["Low","High"]))].copy()
    sub = set_condition_order(sub)
    res = fit_mixedlm(sub, "value ~ C(condition) * C(pair_type)", group_col="response_id", vc_scenario=True)
    term = "C(condition)[T.High]:C(pair_type)[T.FA]"
    p = float(res.pvalues.get(term, np.nan))
    est = float(res.params.get(term, np.nan))

    def boot_func(boot):
        bres = fit_mixedlm(boot, "value ~ C(condition) * C(pair_type)", group_col="response_id", vc_scenario=True)
        return float(bres.params.get(term, np.nan))

    lo, hi = cluster_bootstrap_ci(sub, boot_func, cluster_col="response_id", n_boot=800, seed=123)
    return {"metric": metric_name, "p_raw": p, "estimate": est, "ci_low": lo, "ci_high": hi}

dir_rows = [
    directional_test(theme_long, "theme"),
    directional_test(lang_long, "language"),
    directional_test(detail_long, "detail_words"),
]

stance_dir = stance_long[stance_long["condition"].isin(["Low","High"])].copy()
stance_dir = set_condition_order(stance_dir)
res_stance_dir = fit_mixedlm(stance_dir, "R ~ C(condition)", group_col="response_id", vc_scenario=True)
term_st = "C(condition)[T.High]"
p_st = float(res_stance_dir.pvalues.get(term_st, np.nan))
est_st = float(res_stance_dir.params.get(term_st, np.nan))
def boot_st(boot):
    bres = fit_mixedlm(boot, "R ~ C(condition)", group_col="response_id", vc_scenario=True)
    return float(bres.params.get(term_st, np.nan))
lo_st, hi_st = cluster_bootstrap_ci(stance_dir, boot_st, cluster_col="response_id", n_boot=800, seed=123)
dir_rows.append({"metric":"stance", "p_raw": p_st, "estimate": est_st, "ci_low": lo_st, "ci_high": hi_st})

directional_df = pd.DataFrame(dir_rows)
reject, p_adj, _, _ = multipletests(directional_df["p_raw"].astype(float).values, method="holm", alpha=0.05)
directional_df["p_holm"] = p_adj
directional_df["reject_holm_0.05"] = reject
directional_df.to_csv(OUT_DIR / "holm_family_directional.csv", index=False)
display(directional_df)

# ---------- Final alignment family (Low vs High only) ----------
def final_alignment_test(long_df, metric_name):
    sub = long_df[(long_df["condition"].isin(["Low","High"])) & (long_df["pair_type"]=="FA")].copy()
    sub = set_condition_order(sub)
    res = fit_mixedlm(sub, "value ~ C(condition)", group_col="response_id", vc_scenario=True)
    term = "C(condition)[T.High]"
    p = float(res.pvalues.get(term, np.nan))
    est = float(res.params.get(term, np.nan))
    def boot_func(boot):
        bres = fit_mixedlm(boot, "value ~ C(condition)", group_col="response_id", vc_scenario=True)
        return float(bres.params.get(term, np.nan))
    lo, hi = cluster_bootstrap_ci(sub, boot_func, cluster_col="response_id", n_boot=800, seed=321)
    return {"metric": metric_name, "p_raw": p, "estimate": est, "ci_low": lo, "ci_high": hi}

final_rows = [
    final_alignment_test(theme_long, "theme"),
    final_alignment_test(lang_long, "language"),
    final_alignment_test(detail_long, "detail_words"),
]

stance_fa = stance_long[stance_long["condition"].isin(["Low","High"])].copy()
stance_fa = set_condition_order(stance_fa)
res_stance_fa = fit_mixedlm(stance_fa, "FA ~ C(condition)", group_col="response_id", vc_scenario=True)
p_st = float(res_stance_fa.pvalues.get(term_st, np.nan))
est_st = float(res_stance_fa.params.get(term_st, np.nan))
def boot_st_fa(boot):
    bres = fit_mixedlm(boot, "FA ~ C(condition)", group_col="response_id", vc_scenario=True)
    return float(bres.params.get(term_st, np.nan))
lo_st, hi_st = cluster_bootstrap_ci(stance_fa, boot_st_fa, cluster_col="response_id", n_boot=800, seed=321)
final_rows.append({"metric":"stance", "p_raw": p_st, "estimate": est_st, "ci_low": lo_st, "ci_high": hi_st})

final_df = pd.DataFrame(final_rows)
reject, p_adj, _, _ = multipletests(final_df["p_raw"].astype(float).values, method="holm", alpha=0.05)
final_df["p_holm"] = p_adj
final_df["reject_holm_0.05"] = reject
final_df.to_csv(OUT_DIR / "holm_family_final_alignment.csv", index=False)
display(final_df)

# ---------- Revision magnitude family (Baseline vs Low vs High) omnibus on FF ----------
def revision_omnibus(long_df, metric_name):
    sub = long_df[long_df["pair_type"]=="FF"].copy()
    sub = set_condition_order(sub)
    res = fit_mixedlm(sub, "value ~ C(condition)", group_col="response_id", vc_scenario=True)
    p = wald_omnibus_condition(res, ["C(condition)[T.High]", "C(condition)[T.Baseline]"])
    return {"metric": metric_name, "p_raw": p,
            "estimate_high_vs_low": float(res.params.get("C(condition)[T.High]", np.nan)),
            "estimate_baseline_vs_low": float(res.params.get("C(condition)[T.Baseline]", np.nan))}

rev_rows = [
    revision_omnibus(theme_long, "theme"),
    revision_omnibus(lang_long, "language"),
    revision_omnibus(detail_long, "detail_words"),
]

stance_rev = stance_long.copy()
stance_rev = set_condition_order(stance_rev)
res_stance_rev = fit_mixedlm(stance_rev, "FF ~ C(condition)", group_col="response_id", vc_scenario=True)
p_st_omni = wald_omnibus_condition(res_stance_rev, ["C(condition)[T.High]", "C(condition)[T.Baseline]"])
rev_rows.append({"metric":"stance", "p_raw": p_st_omni,
                 "estimate_high_vs_low": float(res_stance_rev.params.get("C(condition)[T.High]", np.nan)),
                 "estimate_baseline_vs_low": float(res_stance_rev.params.get("C(condition)[T.Baseline]", np.nan))})

revision_df = pd.DataFrame(rev_rows)
reject, p_adj, _, _ = multipletests(revision_df["p_raw"].astype(float).values, method="holm", alpha=0.05)
revision_df["p_holm"] = p_adj
revision_df["reject_holm_0.05"] = reject
revision_df.to_csv(OUT_DIR / "holm_family_revision_omnibus.csv", index=False)
display(revision_df)

print("Saved family-wise Holm tables to outputs/.")


### Revision magnitude post-hoc pairwise comparisons (Baseline/Low/High)

For the **revision magnitude** family (First→Final / `FF`), we run **pairwise post-hoc comparisons**
for each modality:

- High vs Low  
- Baseline vs Low  
- High vs Baseline  

We apply **Holm correction within each modality** across these 3 pairwise p-values.

Outputs:
- `outputs/revision_posthoc_pairwise.csv`


In [ ]:
# Post-hoc pairwise comparisons for revision magnitude (FF only) with Holm within-modality

from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# This cell assumes the previous Holm-by-family cell has already run and created:
# theme_long, lang_long, detail_long, stance_long, and helper functions:
# set_condition_order, fit_mixedlm, cluster_bootstrap_ci.

def mixedlm_ff_posthoc(long_df, metric_name):
    # Fit FF-only mixed model and compute 3 pairwise contrasts with bootstrap CIs.
    sub = long_df[long_df["pair_type"]=="FF"].copy()
    sub = set_condition_order(sub)

    res = fit_mixedlm(sub, "value ~ C(condition)", group_col="response_id", vc_scenario=True)

    # Low is reference: High-Low and Baseline-Low
    b_hl = float(res.params.get("C(condition)[T.High]", np.nan))
    b_bl = float(res.params.get("C(condition)[T.Baseline]", np.nan))
    p_hl = float(res.pvalues.get("C(condition)[T.High]", np.nan))
    p_bl = float(res.pvalues.get("C(condition)[T.Baseline]", np.nan))

    # High vs Baseline = (High-Low) - (Baseline-Low)
    idx = list(res.params.index)
    w = np.zeros(len(idx))
    if "C(condition)[T.High]" in idx:
        w[idx.index("C(condition)[T.High]")] = 1.0
    if "C(condition)[T.Baseline]" in idx:
        w[idx.index("C(condition)[T.Baseline]")] = -1.0
    try:
        ttest = res.t_test(w)
        b_hb = float(ttest.effect)
        p_hb = float(ttest.pvalue)
    except Exception:
        b_hb = np.nan
        p_hb = np.nan

    # Bootstrap CIs
    def boot_param(boot, coef_name):
        bres = fit_mixedlm(boot, "value ~ C(condition)", group_col="response_id", vc_scenario=True)
        return float(bres.params.get(coef_name, np.nan))

    def boot_high_vs_baseline(boot):
        bres = fit_mixedlm(boot, "value ~ C(condition)", group_col="response_id", vc_scenario=True)
        bh = float(bres.params.get("C(condition)[T.High]", np.nan))
        bb = float(bres.params.get("C(condition)[T.Baseline]", np.nan))
        return bh - bb

    lo_hl, hi_hl = cluster_bootstrap_ci(
        sub, lambda b: boot_param(b, "C(condition)[T.High]"),
        cluster_col="response_id", n_boot=800, seed=2024
    )
    lo_bl, hi_bl = cluster_bootstrap_ci(
        sub, lambda b: boot_param(b, "C(condition)[T.Baseline]"),
        cluster_col="response_id", n_boot=800, seed=2025
    )
    lo_hb, hi_hb = cluster_bootstrap_ci(
        sub, boot_high_vs_baseline,
        cluster_col="response_id", n_boot=800, seed=2026
    )

    out = pd.DataFrame([
        {"metric": metric_name, "contrast": "High vs Low", "estimate": b_hl, "p_raw": p_hl, "ci_low": lo_hl, "ci_high": hi_hl},
        {"metric": metric_name, "contrast": "Baseline vs Low", "estimate": b_bl, "p_raw": p_bl, "ci_low": lo_bl, "ci_high": hi_bl},
        {"metric": metric_name, "contrast": "High vs Baseline", "estimate": b_hb, "p_raw": p_hb, "ci_low": lo_hb, "ci_high": hi_hb},
    ])

    # Holm within modality (3 comparisons)
    pvals = out["p_raw"].astype(float).fillna(1.0).values
    reject, p_adj, _, _ = multipletests(pvals, alpha=0.05, method="holm")
    out["p_holm_within_metric"] = p_adj
    out["reject_holm_0.05"] = reject
    return out

posthoc_tables = [
    mixedlm_ff_posthoc(theme_long, "theme"),
    mixedlm_ff_posthoc(lang_long, "language"),
    mixedlm_ff_posthoc(detail_long, "detail_words"),
]

# Stance FF post-hoc (stance_long has FF directly)
stance_ff = stance_long.copy()
stance_ff = set_condition_order(stance_ff)
res_st = fit_mixedlm(stance_ff, "FF ~ C(condition)", group_col="response_id", vc_scenario=True)

b_hl = float(res_st.params.get("C(condition)[T.High]", np.nan))
b_bl = float(res_st.params.get("C(condition)[T.Baseline]", np.nan))
p_hl = float(res_st.pvalues.get("C(condition)[T.High]", np.nan))
p_bl = float(res_st.pvalues.get("C(condition)[T.Baseline]", np.nan))

idx = list(res_st.params.index)
w = np.zeros(len(idx))
if "C(condition)[T.High]" in idx:
    w[idx.index("C(condition)[T.High]")] = 1.0
if "C(condition)[T.Baseline]" in idx:
    w[idx.index("C(condition)[T.Baseline]")] = -1.0
try:
    ttest = res_st.t_test(w)
    b_hb = float(ttest.effect)
    p_hb = float(ttest.pvalue)
except Exception:
    b_hb = np.nan
    p_hb = np.nan

def boot_st_coef(boot, coef_name):
    bres = fit_mixedlm(boot, "FF ~ C(condition)", group_col="response_id", vc_scenario=True)
    return float(bres.params.get(coef_name, np.nan))

def boot_st_hb(boot):
    bres = fit_mixedlm(boot, "FF ~ C(condition)", group_col="response_id", vc_scenario=True)
    bh = float(bres.params.get("C(condition)[T.High]", np.nan))
    bb = float(bres.params.get("C(condition)[T.Baseline]", np.nan))
    return bh - bb

lo_hl, hi_hl = cluster_bootstrap_ci(stance_ff, lambda b: boot_st_coef(b, "C(condition)[T.High]"), cluster_col="response_id", n_boot=800, seed=3031)
lo_bl, hi_bl = cluster_bootstrap_ci(stance_ff, lambda b: boot_st_coef(b, "C(condition)[T.Baseline]"), cluster_col="response_id", n_boot=800, seed=3032)
lo_hb, hi_hb = cluster_bootstrap_ci(stance_ff, boot_st_hb, cluster_col="response_id", n_boot=800, seed=3033)

stance_posthoc = pd.DataFrame([
    {"metric": "stance", "contrast": "High vs Low", "estimate": b_hl, "p_raw": p_hl, "ci_low": lo_hl, "ci_high": hi_hl},
    {"metric": "stance", "contrast": "Baseline vs Low", "estimate": b_bl, "p_raw": p_bl, "ci_low": lo_bl, "ci_high": hi_bl},
    {"metric": "stance", "contrast": "High vs Baseline", "estimate": b_hb, "p_raw": p_hb, "ci_low": lo_hb, "ci_high": hi_hb},
])

pvals = stance_posthoc["p_raw"].astype(float).fillna(1.0).values
reject, p_adj, _, _ = multipletests(pvals, alpha=0.05, method="holm")
stance_posthoc["p_holm_within_metric"] = p_adj
stance_posthoc["reject_holm_0.05"] = reject

posthoc_tables.append(stance_posthoc)

revision_posthoc = pd.concat(posthoc_tables, ignore_index=True)
revision_posthoc.to_csv(OUT_DIR / "revision_posthoc_pairwise.csv", index=False)

print("Saved:", OUT_DIR / "revision_posthoc_pairwise.csv")
display(revision_posthoc)
